In [ ]:
import time 
from getpass import getuser
muid = getuser()
inicio = time.perf_counter()
import pandas as pd
import glob
import os
from datetime import timedelta
import numpy as np
from pandas.tseries.offsets import BDay

month="\\Sept 2026"
print(r"C:\Users\{0}\MerckGroup".format(getuser()))

C:\Users\X286384\MerckGroup


In [22]:
import getpass
username = getpass.getuser()
print("Username:", muid)

Username: X286384


## Auxiliar Functions

In [23]:

def clean_rows_based_on_prefix(df: pd.DataFrame, column_name: str) -> pd.DataFrame:
    """
    Cleans the DataFrame by removing rows where the specified column does not contain
    elements starting with 'M*', 'MX*', 'C*', or 'tbc'.

    Parameters:
    df (pd.DataFrame): The DataFrame to clean.
    column_name (str): The name of the column to check for prefixes.

    Returns:
    pd.DataFrame: The cleaned DataFrame with rows removed where the specified column
                  does not start with the specified prefixes.
    """
    # Define the prefixes to check
    prefixes = ('M', 'MX', 'C', 'tbc')
    
    # Create a mask for rows that start with the specified prefixes
    mask = df[column_name].str.startswith(prefixes, na=False)
    
    # Filter the DataFrame based on the mask
    cleaned_df = df[mask]
    
    return cleaned_df


def set_fin_column(df: pd.DataFrame, column_name_1: str, column_name_2: str) -> pd.DataFrame:
    """
    Compares the specified columns in the DataFrame.
    Creates a new column 'FIN' which takes the value of 'column_name_1' if it is greater,
    otherwise takes the value of 'column_name_2'. If one of the columns is empty,
    it automatically selects the other one.
 
    Parameters:
    df (pd.DataFrame): The DataFrame to modify.
    column_name_1 (str): The name of the first column to compare.
    column_name_2 (str): The name of the second column to compare.
 
    Returns:
    pd.DataFrame: The modified DataFrame with the new 'FIN' column.
    """
    # Fill NaN values in the columns to avoid comparison issues
    df[column_name_1] = df[column_name_1].fillna(df[column_name_2])
    df[column_name_2] = df[column_name_2].fillna(df[column_name_1])
    # Create the 'FIN' column based on the comparison
    df['FIN'] = df[column_name_1].where( pd.to_datetime(df[column_name_1], errors='coerce') >  pd.to_datetime(df[column_name_2], errors='coerce'), df[column_name_2])
    return df


##### SÓLO SÓLIDOS Y TIROIDEOS #############
def merge_fabri_to_acondi(df_acondi, df_fabri, family,liberacion_granel):
    # Merge the two DataFrames on 'LOTE DE GRANEL'
    df_acondi["LOTE PT"]=df_acondi["LOTE PT"].apply(lambda x: x.strip())
    df_fabri["LOTE DE GRANEL"]=df_fabri["LOTE DE GRANEL"].apply(lambda x: x.strip())
    merged_df = df_acondi.merge(
        df_fabri[['LOTE DE GRANEL', 'FIN FABRI PLAN', 'PR_ENTREGA MUESTRA PLAN', 'ENTREGA ÓRDEN ANALÍTICA PLAN',
                  'SKU', 'DESCRIPCIÓN', 'PR_ENTREGA EXPEDIENTE GR PLAN', liberacion_granel,
                   "GR_CANTIDAD", "GR_CANTIDAD C/YIELD","ORDEN GR","RECURSO GR" ]],
        on='LOTE DE GRANEL',
        how='outer'  # Use 'outer' to include all records from both DataFrames
    )
    
    merged_df["BT_analysis"]="Si va"

    
    claves_no_va=["MX11900221", "MX11900223", "MX11900077", "3185850012", "3184630003",
    "3184530004", "3184720003", "3186220002", "3194520015", "MX11900325",
    "MX11900483", "MX11900449", "3186280027", "3186280018", "3214230003",
    "3186280025","3184630002", "3186280029", "3186280030", "3185640022",
    "3185640021", "3951770115", "3951770114", "3951030071", "3951090007",
    "3951030070", "3951780112","3951090006" ]
 

    # --- Case 1: family == "solids" ---
    if family == "solids":
        # Identify rows where SKU_x is missing
        missing_mask = merged_df["SKU_x"].isna() | (merged_df["SKU_x"] == "")

        # Replace SKU_x and DESCRIPCIÓN_x with values from _y
        merged_df.loc[missing_mask, "SKU_x"] = merged_df.loc[missing_mask, "SKU_y"]
        merged_df.loc[missing_mask, "DESCRIPCIÓN_x"] = merged_df.loc[missing_mask, "DESCRIPCIÓN_y"]
        condition = (
            merged_df["SKU_y"].isin(claves_no_va) &
            (merged_df["SKU_x"].isna() | (merged_df["SKU_x"].astype(str).str.strip() == ""))
        )
        merged_df.loc[condition, "BT_analysis"] = "No va"

    # --- Case 2: family == "tyroids" ---
    elif family == "tyroids":
        # Identify rows where SKU_x is missing
        missing_mask = merged_df["SKU_x"].isna() | (merged_df["SKU_x"] == "")

        # Replace SKU_x and DESCRIPCIÓN_x with values from _y
        merged_df.loc[missing_mask, "SKU_x"] = merged_df.loc[missing_mask, "SKU_y"]
        merged_df.loc[missing_mask, "DESCRIPCIÓN_x"] = merged_df.loc[missing_mask, "DESCRIPCIÓN_y"]

        # If description_y does not contain MMSA or BRA → "No va"
        invalid_desc_mask = ~merged_df["DESCRIPCIÓN_y"].str.contains("MMSA|BRA", na=False)

        merged_df.loc[missing_mask & invalid_desc_mask, "BT_analysis"] = "No va"
        
    df_acondi_cosolidated=merged_df   
    df_acondi_cosolidated["LOTE DE GRANEL 2"]="No Aplica"
    df_acondi_cosolidated["TYPE_BATCH"]="E2E"

    def classify(row):
        if pd.notna(row['LOTE DE GRANEL']) and row['LOTE DE GRANEL'].startswith('MX'):

            return 'IMPORTED'
        return row['TYPE_BATCH']
    
    
    df_acondi_cosolidated["TYPE_BATCH"] = df_acondi_cosolidated.apply(classify, axis=1)

    df_acondi_cosolidated.rename(columns={'SKU_x':"SKU",'DESCRIPCIÓN_x':"DESCRIPCIÓN"},inplace=True)
    
    print(df_acondi_cosolidated.keys())
    return df_acondi_cosolidated

##### SÓLO LIQUIDOS #############
def merge_and_process_batches(df_acondi, df_fabri):
    # Match LOTE DE GRANEL with LOTE DE GRANEL in df_acondi
    df_acondi["LOTE PT"]=df_acondi["LOTE PT"].apply(lambda x: x.strip())
    df_fabri["LOTE DE GRANEL"]=df_fabri["LOTE DE GRANEL"].apply(lambda x: x.strip())
    
    merged_1 = df_acondi.merge(
        df_fabri[['LOTE DE GRANEL', 'FIN FABRI PLAN', 'PR_ENTREGA MUESTRA PLAN', 
                   'ENTREGA ÓRDEN ANALÍTICA PLAN', 'PL_STATUS', 'PR_ENTREGA EXPEDIENTE GR PLAN',
                    "GR_CANTIDAD", "GR_CANTIDAD C/YIELD", "ORDEN GR", "RECURSO GR"]],
        left_on='LOTE DE GRANEL',
        right_on='LOTE DE GRANEL',
        how='left'
    )
    
    # Rename columns with suffix _1
    merged_1 = merged_1.rename(columns={
        'FIN FABRI PLAN': 'FIN FABRI PLAN_1',
        'PR_ENTREGA MUESTRA PLAN': 'PR_ENTREGA MUESTRA PLAN_1',
        'ENTREGA ÓRDEN ANALÍTICA PLAN': 'ENTREGA ÓRDEN ANALÍTICA PLAN_1',
        'PL_STATUS': 'PL_STATUS_1',
        'PR_ENTREGA EXPEDIENTE GR PLAN': 'PR_ENTREGA EXPEDIENTE GR PLAN_1'
    })

    # Match LOTE DE GRANEL from df_fabri with LOTE DE GRANEL 2 in df_acondi
    merged_2 = merged_1.merge(
        df_fabri[['LOTE DE GRANEL', 'FIN FABRI PLAN', 'PR_ENTREGA MUESTRA PLAN', 
                   'ENTREGA ÓRDEN ANALÍTICA PLAN', 'PL_STATUS', 'PR_ENTREGA EXPEDIENTE GR PLAN']],
        left_on='LOTE DE GRANEL 2',
        right_on='LOTE DE GRANEL',
        how='left'
    )

    # Rename columns with suffix _2
    merged_2 = merged_2.rename(columns={
        'FIN FABRI PLAN': 'FIN FABRI PLAN_2',
        'PR_ENTREGA MUESTRA PLAN': 'PR_ENTREGA MUESTRA PLAN_2',
        'ENTREGA ÓRDEN ANALÍTICA PLAN': 'ENTREGA ÓRDEN ANALÍTICA PLAN_2',
        'PL_STATUS': 'PL_STATUS_2',
        'PR_ENTREGA EXPEDIENTE GR PLAN': 'PR_ENTREGA EXPEDIENTE GR PLAN_2',
        'LOTE DE GRANEL_x':'LOTE DE GRANEL'
    })

    # Compare the columns and create new columns with the most recent date
    # Update df_acondi with the new columns
    merged_2['FIN FABRI PLAN'] =  set_fin_column(df=merged_2,column_name_1='FIN FABRI PLAN_1',
                                                              column_name_2='FIN FABRI PLAN_2')["FIN"]
    merged_2['PR_ENTREGA MUESTRA PLAN'] = set_fin_column(df=merged_2,column_name_1='PR_ENTREGA MUESTRA PLAN_1',
                                                                      column_name_2='PR_ENTREGA MUESTRA PLAN_2')["FIN"]
    merged_2['ENTREGA ÓRDEN ANALÍTICA PLAN'] = set_fin_column(df=merged_2,column_name_1='ENTREGA ÓRDEN ANALÍTICA PLAN_1',
                                                                      column_name_2='ENTREGA ÓRDEN ANALÍTICA PLAN_2')["FIN"]
    
    merged_2['PR_ENTREGA EXPEDIENTE GR PLAN'] = set_fin_column(df=merged_2,column_name_1='PR_ENTREGA EXPEDIENTE GR PLAN_1',
                                                                      column_name_2='PR_ENTREGA EXPEDIENTE GR PLAN_2')["FIN"]
    print(merged_2.keys())
    merged_2
    
    df_acondi_2=merged_2[['SKU', 'DESCRIPCIÓN', 'LOTE PT', 'LOTE DE GRANEL', 'LOTE DE GRANEL 2', 'PL_STATUS_x' ,'FECHA REQ', 'FECHA OTIF',
                        'FIN FABRI PLAN', 'PR_ENTREGA MUESTRA PLAN',  'PR_ENTREGA EXPEDIENTE GR PLAN','ENTREGA ÓRDEN ANALÍTICA PLAN',
                        'ACO_FIN ACONDI PLAN','PT_ENTREGA EXPEDIENTE PLAN','FECHA LIBERACION V0', "GR_CANTIDAD", "GR_CANTIDAD C/YIELD", 
                        "PT_CANTIDAD", "PT_CANTIDAD C/YIELD", "ORDEN GR", "ORDEN PT", "RECURSO PT", "RECURSO GR"]]
    df_acondi_2.rename(columns={'PL_STATUS_x':"PL_STATUS"})
    df_acondi_2["TYPE_BATCH"]="E2E"
    df_acondi_2["BT_analysis"]="Si va"

    return df_acondi_2, merged_2

def filter_by_date(df, year=None, months=None):
    # Ensure that the date columns are in datetime format
    df['FECHA LIBERACION V0'] = pd.to_datetime(df['FECHA LIBERACION V0'], errors='coerce')
    df['FIN FABRI PLAN'] = pd.to_datetime(df['FIN FABRI PLAN'], errors='coerce')

    # Filter by year
    if year is not None:
        df = df[(df['FECHA LIBERACION V0'].dt.year.isin(year)) | (df['FIN FABRI PLAN'].dt.year.isin(year))]

    # Filter by months
    if months is not None:
        df = df[(df['FECHA LIBERACION V0'].dt.month.isin(months)) | (df['FIN FABRI PLAN'].dt.month.isin(months))]

    return df  # Return only the specified columns

# Example usage
# filtered_df = filter_by_date(df_fabri, year=2023, months=[5, 6])  # Filter for May and June 2023
# filtered_df = filter_by_date(df_fabri, year=2023)  # Filter for the entire year 2023
# filtered_df = filter_by_date(df_fabri, months=[5, 6])  # Filter for May and June of any year
def update_lote_pt(bt_solidos, fabri_solidos):
    # Ensure the columns are named correctly
    bt_solidos = bt_solidos.copy()
    fabri_solidos = fabri_solidos.copy()
    
    # Iterate through the fabri_solidos DataFrame
    for index, row in fabri_solidos.iterrows():
        lote_granel_value    = row['Lote Granel']
        lote_value           = row['Lote']
        product_value        = row['Producto']
        fecha_dictamen_value = row['FECHA DE DICTAMEN']
        clave_value          = row['Clave']
        
        # Check if Lote Granel matches with LOTE DE GRANEL in BT_solidos
        matches = bt_solidos[bt_solidos['LOTE DE GRANEL'] == lote_granel_value]
        
        if not matches.empty:
            # Update the LOTE PT column in BT_solidos with the Lote value from fabri_solidos
            bt_solidos.loc[bt_solidos['LOTE DE GRANEL'] == lote_granel_value, 'LOTE PT'] = lote_value
            
            # Update the DESCRIPCIÓN column in BT_solidos with the PRODUCT value from fabri_solidos
            bt_solidos.loc[bt_solidos['LOTE DE GRANEL'] == lote_granel_value, 'DESCRIPCIÓN'] = product_value
            
            # Update the FECHA LIBERACION V0 column in BT_solidos with the FECHA DE DICTAMEN value from fabri_solidos
            bt_solidos.loc[bt_solidos['LOTE DE GRANEL'] == lote_granel_value, 'FECHA LIBERACION V0'] = fecha_dictamen_value
            bt_solidos.loc[bt_solidos['LOTE DE GRANEL'] == lote_granel_value, 'SKU'] = clave_value
            
    return bt_solidos


# Sample DataFrames (you would replace this with your actual DataFrames)
# merged_df = pd.DataFrame(...)  # Your merged DataFrame
# BatchTracker_df = pd.DataFrame(...)  # Your BatchTracker DataFrame

# Function to create Deviation_Info based on matches
def create_deviation_info(row):
    deviation_info = []
    
    # Check for matches in 'LOTE PT'
    if row['LOTE PT'] in merged_df['BatchNumberUnique'].values:
        matched_rows = merged_df[merged_df['BatchNumberUnique'] == row['LOTE PT']]
        for _, matched_row in matched_rows.iterrows():
            deviation_info.append(
                f"Lote PT: {matched_row['PR ID']} - {matched_row['Title']} \n"
                f"Due Date - Assigned to: {matched_row['Record State']} - {matched_row['HC Due Date']} - {matched_row['Assigned to']}\n"
                f"                                                                 "
            )
    
    # Check for matches in 'LOTE DE GRANEL'
    if row['LOTE DE GRANEL'] in merged_df['BatchNumberUnique'].values:
        matched_rows = merged_df[merged_df['BatchNumberUnique'] == row['LOTE DE GRANEL']]
        for _, matched_row in matched_rows.iterrows():
            deviation_info.append(
                f"Lote GRANEL: {matched_row['PR ID']} - {matched_row['Title']} \n"
                f"Due Date - Assigned to: {matched_row['Record State']} - {matched_row['HC Due Date']} - {matched_row['Assigned to']}\n"
                f"                                                                 "
            )
    
    # Check for matches in 'LOTE DE GRANEL 2\n(Solo Liq. Esteriles)'
    if row['LOTE DE GRANEL 2'] in merged_df['BatchNumberUnique'].values:
        matched_rows = merged_df[merged_df['BatchNumberUnique'] == row['LOTE DE GRANEL 2']]
        for _, matched_row in matched_rows.iterrows():
            deviation_info.append(
                f"Lote GRANEL 2 - LIQ: {matched_row['PR ID']} - {matched_row['Title']} \n"
                f"Due Date - Assigned to: {matched_row['Record State']} - {matched_row['HC Due Date']} - {matched_row['Assigned to']}\n"
                f"                                                                 "
            )
    
    return "\n".join(deviation_info) if deviation_info else ""


# Function to create Deviation Due Date based on matches
def create_deviation_due_date(row):
    due_dates = []

    # Check for matches in 'LOTE PT'
    if row['LOTE PT'] in merged_df['BatchNumberUnique'].values:
        matched_rows = merged_df[merged_df['BatchNumberUnique'] == row['LOTE PT']]
        due_dates.extend(matched_rows['HC Due Date'].dropna().tolist())

    # Check for matches in 'LOTE DE GRANEL'
    if row['LOTE DE GRANEL'] in merged_df['BatchNumberUnique'].values:
        matched_rows = merged_df[merged_df['BatchNumberUnique'] == row['LOTE DE GRANEL']]
        due_dates.extend(matched_rows['HC Due Date'].dropna().tolist())

    # Check for matches in 'LOTE DE GRANEL 2\n(Solo Liq. Esteriles)'
    if row['LOTE DE GRANEL 2'] in merged_df['BatchNumberUnique'].values:
        matched_rows = merged_df[merged_df['BatchNumberUnique'] == row['LOTE DE GRANEL 2']]
        due_dates.extend(matched_rows['HC Due Date'].dropna().tolist())

    # Filter the due_dates to ensure only valid dates are considered
    due_dates = [date for date in due_dates if isinstance(date, (pd.Timestamp, str)) and date != '']

    # If there are due dates, find the earliest one
    if due_dates:
        earliest_due_date = min(pd.to_datetime(due_dates, format="mixed", dayfirst=True, yearfirst=False))
        return earliest_due_date.strftime('%d/%m/%Y')  # Format the date as DD/MM/YYYY

    return ""

# Function to create Deviation Due Date based on matches
def create_deviation_closure_date(row,):
    due_dates = []

    # Check for matches in 'LOTE PT'
    if row['LOTE PT'] in merged_df['BatchNumberUnique'].values:
        matched_rows = merged_df[merged_df['BatchNumberUnique'] == row['LOTE PT']]
        due_dates.extend(matched_rows['Date Closed'].dropna().tolist())

    # Check for matches in 'LOTE DE GRANEL'
    if row['LOTE DE GRANEL'] in merged_df['BatchNumberUnique'].values:
        matched_rows = merged_df[merged_df['BatchNumberUnique'] == row['LOTE DE GRANEL']]
        due_dates.extend(matched_rows['Date Closed'].dropna().tolist())

    # Check for matches in 'LOTE DE GRANEL 2\n(Solo Liq. Esteriles)'
    if row['LOTE DE GRANEL 2'] in merged_df['BatchNumberUnique'].values:
        matched_rows = merged_df[merged_df['BatchNumberUnique'] == row['LOTE DE GRANEL 2']]
        due_dates.extend(matched_rows['Date Closed'].dropna().tolist())

    # Filter the due_dates to ensure only valid dates are considered
    due_dates = [date for date in due_dates if isinstance(date, (pd.Timestamp, str)) and date != '']

    # If there are due dates, find the oldest one
    if due_dates:
        earliest_due_date = max(pd.to_datetime(due_dates, format="mixed", dayfirst=True, yearfirst=False))
        return earliest_due_date.strftime('%d/%m/%Y')  # Format the date as DD/MM/YYYY

    return ""


## Data Extraction from different sources



In [24]:
###################   BATCH TRACKER                 #######################################################################
from getpass import getuser
base_address = r"C:\Users\{0}\MerckGroup".format(getuser())
path_BatchTracker= r"{0}\Seguimiento Servicio GMS P&G-BP - BATCHTRACKER\BASE BATCHTRACKER\Batch Tracker.xlsx".format(base_address)
BatchTracker_df= pd.read_excel(path_BatchTracker, header=1, sheet_name="BatchTracker")

####################   REVISORES ACONDI             ######################################################################
path_acondi_tiroideos_QA_piso = 'C:\\Users\\{0}\\MerckGroup\\EQ REVISORES - Equipo Revisor Acondicionamiento\\Revisión Hormonales Ene 2023.xlsx'.format(muid)
path_acondi_liquidos_QA_piso  = 'C:\\Users\\{0}\\MerckGroup\\EQ REVISORES - Equipo Revisor Acondicionamiento\\Revisión Líquidos Ene 2023.xlsx'.format(muid)
path_acondi_solidos_QA_piso   = 'C:\\Users\\{0}\\MerckGroup\\EQ REVISORES - Equipo Revisor Acondicionamiento\\Revisión Sólidos Ene 2023.xlsx'.format(muid)

acondi_tiroideos_QA_piso  = pd.read_excel(path_acondi_tiroideos_QA_piso,sheet_name="Hormonales").copy()
acondi_tiroideos_QA_piso['ÁREA']="Tiroideos"
acondi_liquidos_QA_piso   = pd.read_excel(path_acondi_liquidos_QA_piso,  sheet_name="Líquidos").copy()
acondi_liquidos_QA_piso['ÁREA']="Líquidos"
acondi_solidos_QA_piso = pd.read_excel(path_acondi_solidos_QA_piso,  sheet_name="Sólidos").copy()
acondi_solidos_QA_piso['ÁREA']="Sólidos"
#####################  REVISORES FABRI             #######################################################################

path_fabri_tiroideos_QA_piso = r'C:\Users\{0}\MerckGroup\RFT PRODUCCIÓN - Documents\RFT TIROIDEOS\Lista Ordenes EN PROCESO.xlsx'.format(muid)
path_fabri_liquidos_QA_piso  = r'C:\Users\{0}\MerckGroup\Expedientes fabricación - General\Lead Time.xlsx'.format(muid)
path_fabri_solidos_QA_piso   = r'C:\Users\{0}\MerckGroup\RFT PRODUCCIÓN - Documents\RFT SOLIDOS\RFT SÓLIDOS.xlsm'.format(muid)

fabri_tiroideos_QA_piso  = pd.read_excel(path_fabri_tiroideos_QA_piso,sheet_name="OrdenesTiroideos").copy()
fabri_tiroideos_QA_piso['ÁREA']="Tiroideos"
fabri_liquidos_QA_piso   = pd.read_excel(path_fabri_liquidos_QA_piso,  sheet_name="Liquidos").copy()
fabri_liquidos_QA_piso['ÁREA']="Líquidos"
fabri_solidos_QA_piso = pd.read_excel(path_fabri_solidos_QA_piso,  sheet_name="Registro",header=4).copy()
fabri_solidos_QA_piso['ÁREA']="Sólidos"

###################    QC                            #########################################################################

path_QC_general=r"C:\Users\{0}\MerckGroup\Control de calidad Mexico - Monitoreo de liberaciones de PT 2022\Copy of MONITOREO DE LIBERACIONES PT 2022.xlsx".format(muid)
path_QC_tiroides=r"C:\Users\{0}\MerckGroup\LIBERACIONES EUTIROX - Documents\General\MONITOREO DE LIBERACIONES HORMONALES ENERO 2022.xlsx".format(muid)

QC_general  = pd.read_excel(path_QC_general,sheet_name="BASE GENERAL 2022-2023").copy()
QC_tiroides  = pd.read_excel(path_QC_tiroides,sheet_name="BASE GENERAL").copy()





############# QA Dictamen                      ################################################################################



c:\Users\x286384\MerckGroup\Digitalización - Documents\DigitalProjectsLibrary\Planning\Joe\PlanningTranslation\venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
c:\Users\x286384\MerckGroup\Digitalización - Documents\DigitalProjectsLibrary\Planning\Joe\PlanningTranslation\venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
c:\Users\x286384\MerckGroup\Digitalización - Documents\DigitalProjectsLibrary\Planning\Joe\PlanningTranslation\venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
c:\Users\x286384\MerckGroup\Digitalización - Documents\DigitalProjectsLibrary\Planning\Joe\PlanningTranslation\venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and 

In [26]:
path_supply_chain    =r"C:\Users\{0}\MerckGroup\Supply Chain - Planes de Producción 2026{1}".format(muid,month)
directory            =r"C:\Users\{0}\MerckGroup\Supply Chain - Planes de Producción 2026{1}".format(muid, month)  # Change this to your directory



############## Programas de Planeacion         ################################################################################

path_plan_liquidos = glob.glob(os.path.join(directory, '*FABRICACIÓN-ACONDICIONAMIENTO LÍQUIDOS*.xlsx'))[0]
path_plan_solidos_fabri   = glob.glob(os.path.join(directory, '*PLAN FABRICACIÓN SOLIDOS*.xlsx'))[0] 
path_plan_solidos_acondi  = glob.glob(os.path.join(directory, '*PLAN ACONDICIONAMIENTO SOLIDOS*.xlsx'))[0]  
path_plan_tiroides_acondi = glob.glob(os.path.join(directory, '*PLAN ACONDICIONAMIENTO HORMONALES*.xlsx'))[0]
path_plan_tiroides_fabri  = glob.glob(os.path.join(directory, '*PLAN FABRICACIÓN HORMONALES*.xlsx'))[0] 

print(path_plan_liquidos)
print(path_plan_solidos_fabri)
print(path_plan_solidos_acondi)
print(path_plan_tiroides_acondi)
print(path_plan_tiroides_fabri)


C:\Users\X286384\MerckGroup\Supply Chain - Planes de Producción 2026\Octubre 2026\DRAFT FABRICACIÓN-ACONDICIONAMIENTO LÍQUIDOS OCTUBRE.xlsx
C:\Users\X286384\MerckGroup\Supply Chain - Planes de Producción 2026\Octubre 2026\PLAN FABRICACIÓN SOLIDOS Draft Octubre.xlsx
C:\Users\X286384\MerckGroup\Supply Chain - Planes de Producción 2026\Octubre 2026\PLAN ACONDICIONAMIENTO SOLIDOS DRAFT Octubre.xlsx
C:\Users\X286384\MerckGroup\Supply Chain - Planes de Producción 2026\Octubre 2026\Draft PLAN ACONDICIONAMIENTO HORMONALES   Oct26 v1.xlsx
C:\Users\X286384\MerckGroup\Supply Chain - Planes de Producción 2026\Octubre 2026\Draft PLAN FABRICACIÓN HORMONALES   Oct26.xlsx


In [27]:


### Líquidos

plan_liquidos_fabri_NE=pd.read_excel(path_plan_liquidos, sheet_name="LIQUIDOS NE FABRI", header=3)
plan_liquidos_fabri_DC=pd.read_excel(path_plan_liquidos, sheet_name="DC fab", header=4)
plan_liquidos_fabri_AM=pd.read_excel(path_plan_liquidos, sheet_name="AMPOLLETAS FABRICACIÓN", header=4)

plan_liquidos_acondi_NE=pd.read_excel(path_plan_liquidos, sheet_name="LIQUIDOS NE EMPAQUE", header=4)
plan_liquidos_acondi_DC=pd.read_excel(path_plan_liquidos, sheet_name="DC EMPAQUE", header=5)
plan_liquidos_acondi_AM=pd.read_excel(path_plan_liquidos, sheet_name="AMPOLLETAS EMPAQUE", header=5)

### Sólidos
plan_solidos_fabri_TAB12= pd.read_excel(path_plan_solidos_fabri, sheet_name="TAB-12", header=5)
plan_solidos_fabri_TAB13= pd.read_excel(path_plan_solidos_fabri, sheet_name="TAB-13", header=4)
plan_solidos_fabri_TAB19= pd.read_excel(path_plan_solidos_fabri, sheet_name="TAB-19", header=5)
plan_solidos_fabri_TAB24= pd.read_excel(path_plan_solidos_fabri, sheet_name="TAB-24", header=5)
plan_solidos_fabri_ENV= pd.read_excel(path_plan_solidos_fabri, sheet_name="Envasado", header=4)
plan_solidos_fabri_PLV= pd.read_excel(path_plan_solidos_fabri, sheet_name="PLV", header=4)

plan_solidos_acondi_L3= pd.read_excel(path_plan_solidos_acondi, sheet_name="LINEA 3", header=3)
plan_solidos_acondi_L4= pd.read_excel(path_plan_solidos_acondi, sheet_name="LINEA 4", header=3)
plan_solidos_acondi_L5= pd.read_excel(path_plan_solidos_acondi, sheet_name="LINEA 5", header=3)
plan_solidos_acondi_L7= pd.read_excel(path_plan_solidos_acondi, sheet_name="LINEA 7", header=3)
plan_solidos_acondi_L6= pd.read_excel(path_plan_solidos_acondi, sheet_name="LINEA 6 CAR-18", header=3)
plan_solidos_acondi_LC= pd.read_excel(path_plan_solidos_acondi, sheet_name="LINEA CESOL", header=3)
plan_solidos_acondi_LM= pd.read_excel(path_plan_solidos_acondi, sheet_name="LINEA MANUAL", header=3)

### Tiroideos

plan_tiroideos_fabri_NF=pd.read_excel(path_plan_tiroides_fabri, sheet_name="NUEVA FORMULA", header=4)
plan_tiroideos_fabri_VF=pd.read_excel(path_plan_tiroides_fabri, sheet_name="Novotiral", header=4)

plan_tiroideos_acondi_L8=pd.read_excel(path_plan_tiroides_acondi, sheet_name="ASL8", header=5)
plan_tiroideos_acondi_L9=pd.read_excel(path_plan_tiroides_acondi, sheet_name="ASL9", header=5)
plan_tiroideos_acondi_L10=pd.read_excel(path_plan_tiroides_acondi, sheet_name="ASL10", header=5)
plan_tiroideos_acondi_LM=pd.read_excel(path_plan_tiroides_acondi, sheet_name="MAQUILA", header=3)



c:\Users\x286384\MerckGroup\Digitalización - Documents\DigitalProjectsLibrary\Planning\Joe\PlanningTranslation\venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)
c:\Users\x286384\MerckGroup\Digitalización - Documents\DigitalProjectsLibrary\Planning\Joe\PlanningTranslation\venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)
c:\Users\x286384\MerckGroup\Digitalización - Documents\DigitalProjectsLibrary\Planning\Joe\PlanningTranslation\venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)
c:\Users\x286384\MerckGroup\Digitalización - Documents\DigitalProjectsLibrary\Planning\Joe\PlanningTranslation\venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not s

## Trackwise - Deviations

In [28]:
deviations_TW= pd.read_excel("PRODUCT_INFO.xlsx")
details_TW=pd.read_excel("DEVIATIONS_INFO.xlsx")
 
merged_df = pd.merge(deviations_TW, details_TW, left_on="PR ID", right_on="Record ID", how="left")  # Use 'outer', 'left', or 'right' as needed
 
 
len(merged_df)
raw_crossdata= merged_df.to_excel("RawConsolidated.xlsx")
merged_df


c:\Users\x286384\MerckGroup\Digitalización - Documents\DigitalProjectsLibrary\Planning\Joe\PlanningTranslation\venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\x286384\MerckGroup\Digitalización - Documents\DigitalProjectsLibrary\Planning\Joe\PlanningTranslation\venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,PR ID,Grid ID,HC Product Name,HC Item Code #,HC Batch Number,HC Study Number,Record ID,Title,Assigned to,Originator,...,HC Submitted On,HC QA Assessed On,HC Due Date,HC INV/CAPA Approved On,Date Closed,HC Product Related?,HC Root Cause Category,HC Fine Root Cause Category,HC Item Code Number,HC Initial Risk Assessment
0,5342546,1582.0,NaN,NaN,Varios lotes ver Delivery,Varios lotes,5342546,BP Excursion de temperatura durante el traslad...,Jose Vazquez,"Garcia, Jacqueline",...,2026-02-24 13:03:00,2026-02-25 13:59:00,2026-03-26,2026-03-05 17:58:00,2026-03-05 22:42:00,Yes,Machine,Technical Failure,NaN,Minor
1,5190000,1582.0,Erbitux,3023713701,Varios,NaN,5190000,BP Agote de inventario de producto Erbitux / D...,Karina Vergara,"Vergara, Karina",...,2025-11-20 18:31:00,2025-11-20 18:31:00,2025-11-28,2025-11-28 15:24:00,2025-11-28 15:31:00,Yes,Method,Method/Procedure not Defined,NaN,Minor
2,5195574,1582.0,NaN,NaN,Varios,NaN,5195574,Etiqueta de calibración vencida Agosto 2025. D...,Alma Elizabeth Lopez,"Garcia, Jacqueline",...,2025-11-03 16:05:00,2025-11-04 19:04:00,2025-12-03,2025-11-27 11:26:00,2025-11-27 13:14:00,Yes,Method,Missing Detection System,NaN,Minor
3,5332577,1582.0,NaN,Varios,Varios,Varios,5332577,BP Error en la carga de los contenedores FSCU5...,Jose Vazquez,"Garcia, Jacqueline",...,2026-04-20 14:13:00,2026-04-28 13:11:00,2026-03-19,2026-04-28 20:04:00,2026-04-30 15:45:00,Yes,Manpower,Human Factor/Behaviour,NaN,Major
4,5165250,1582.0,NaN,3448410001 RUPILIP 20MG TAB - (30) MEX,PP8672,NaN,5165250,BP - Arte no actualizada para cajilla e instru...,Abraham Hernandez,"Cabello, Antonio",...,2025-10-10 13:48:00,2025-10-13 17:55:00,2025-11-09,2025-10-29 16:16:00,2025-10-30 17:20:00,Yes,Method,Missing Detection System,NaN,Major
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
694,5334692,1582.0,"CONCOR 2,5ACT SUBST FREE TAB_309889",MX11886,NaN,NaN,5334692,Concor 2.5_ 5_10 mg- Discrepancia entre las Es...,Mexico Deviations,"Garcia, Zoraida",...,2026-02-18 20:13:00,2026-02-18 20:14:00,2026-03-20,2026-03-20 14:58:00,2026-03-20 15:02:00,Yes,Other,Not Identified/not Confirmed,NaN,Major
695,5334692,1582.0,CONCOR 5 MG_300803,MX11887,NaN,NaN,5334692,Concor 2.5_ 5_10 mg- Discrepancia entre las Es...,Mexico Deviations,"Garcia, Zoraida",...,2026-02-18 20:13:00,2026-02-18 20:14:00,2026-03-20,2026-03-20 14:58:00,2026-03-20 15:02:00,Yes,Other,Not Identified/not Confirmed,NaN,Major
696,5410025,1582.0,NaN,M60444,NaN,NaN,5410025,MAQ-Incidente-NEUROBION 10 000 DC Lote M60444_...,Montserrat Velazquez,"Lemus, Juan",...,2026-04-14 15:31:00,2026-04-23 12:13:00,2026-05-13,2026-05-13 16:03:00,2026-05-13 16:07:00,Yes,Method,Other,NaN,Minor
697,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaT,NaT,NaT,NaT,NaT,NaN,NaN,NaN,NaN,NaN


In [29]:
import pandas as pd

l_real_batches = []
for i in range(len(merged_df)):
    # Get the batch number, handling NaN values
    batch_number = merged_df["HC Batch Number"].iloc[i]
    
    if pd.isna(batch_number):  # Check if the value is NaN
        real_batch = "MXXX"  # Set to "MXXX" if empty
    else:
        # Remove leading spaces and commas
        batch_number = batch_number.lstrip().replace(',', ' ')
        
        print(i)
        print(batch_number)

        # Process the batch number
        if "MEX Lote" in batch_number:
            real_batch = batch_number.split(":")[-1]
        else: 
            real_batch = batch_number.split(" ")[0]
    
    l_real_batches.append(real_batch.replace(" ", ""))

merged_df["BatchNumberUnique"] = l_real_batches

merged_df = merged_df[['PR ID', 'Grid ID', 'HC Product Name', 'HC Item Code #',
       'HC Batch Number', 'HC Study Number', 'BatchNumberUnique', "Title", 'Record ID', 'Originator',
       'Date Opened', 'Date Closed', 'Assigned to',
       'Record State', 'HC Due Date']]

date_columns = ['Date Opened', 'Date Closed', 'HC Due Date']
# Convert the date column to datetime format
for i in date_columns:
    merged_df[i] = pd.to_datetime(merged_df[i], format="mixed", dayfirst=False, yearfirst=False)

    # Format the date to the desired format
    merged_df[i] = merged_df[i].dt.strftime('%d/%m/%Y')

# Save the DataFrame to an Excel file
merged_df.to_excel("consolidated_TW_report.xlsx", index=False)
merged_df



0
Varios lotes ver Delivery
1
Varios
2
Varios
3
Varios
4
PP8672
5
PP1150
6
PN7592
7
not batch-related
8
MX11900228 AMP DEXABION l M61973
9
MX10120546
10
MX10120546
11
MX10120177
12
MX10119947
13
MX10119945
14
MX10119892
15
MX10119891
16
MX10119677
17
MX10119676
18
MX10119661
19
MX10119659
20
MX10119658
21
MX10119655
22
MX10119654
23
MX10119618
24
MX10119616
25
MX10119553
26
MX10119552
27
MX10119363
28
MX10119320
29
MX10119274
30
MX10119224  (50012932)
31
MX10119218
32
MX10119064
33
MX10119029
34
MX10119028
35
MX10119026
36
MX10118849
37
MX10118848
38
MX10118846
39
MX10118831
40
MX10118759
41
MX10118758
42
MX10118757
43
MX10118756
44
MX10118755
45
MX10118754
46
MX10118753
47
MX10118752
48
MX10118602
49
MX10118599
50
MX10118277
51
MX10118123
52
MX10117937
53
MX10117929
54
MX10117622
55
MX10117621
56
MX10117169
57
MX10117103
58
MX10117051
59
MX10116908
60
MX10116809
61
MX10116808
62
MX10116674
63
MX10116673
64
MX10116546
65
MX10116480
66
MX10116383
67
MX10116382
68
MX10116381
69
MX1011638

,PR ID,Grid ID,HC Product Name,HC Item Code #,HC Batch Number,HC Study Number,BatchNumberUnique,Title,Record ID,Originator,Date Opened,Date Closed,Assigned to,Record State,HC Due Date
0,5342546,1582.0,NaN,NaN,Varios lotes ver Delivery,Varios lotes,Varios,BP Excursion de temperatura durante el traslad...,5342546,"Garcia, Jacqueline",24/02/2026,05/03/2026,Jose Vazquez,Closed,26/03/2026
1,5190000,1582.0,Erbitux,3023713701,Varios,NaN,Varios,BP Agote de inventario de producto Erbitux / D...,5190000,"Vergara, Karina",29/10/2025,28/11/2025,Karina Vergara,Closed,28/11/2025
2,5195574,1582.0,NaN,NaN,Varios,NaN,Varios,Etiqueta de calibración vencida Agosto 2025. D...,5195574,"Garcia, Jacqueline",03/11/2025,27/11/2025,Alma Elizabeth Lopez,Closed,03/12/2025
3,5332577,1582.0,NaN,Varios,Varios,Varios,Varios,BP Error en la carga de los contenedores FSCU5...,5332577,"Garcia, Jacqueline",17/02/2026,30/04/2026,Jose Vazquez,Closed,19/03/2026
4,5165250,1582.0,NaN,3448410001 RUPILIP 20MG TAB - (30) MEX,PP8672,NaN,PP8672,BP - Arte no actualizada para cajilla e instru...,5165250,"Cabello, Antonio",10/10/2025,30/10/2025,Abraham Hernandez,Closed,09/11/2025
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
694,5334692,1582.0,"CONCOR 2,5ACT SUBST FREE TAB_309889",MX11886,NaN,NaN,MXXX,Concor 2.5_ 5_10 mg- Discrepancia entre las Es...,5334692,"Garcia, Zoraida",18/02/2026,20/03/2026,Mexico Deviations,Closed,20/03/2026
695,5334692,1582.0,CONCOR 5 MG_300803,MX11887,NaN,NaN,MXXX,Concor 2.5_ 5_10 mg- Discrepancia entre las Es...,5334692,"Garcia, Zoraida",18/02/2026,20/03/2026,Mexico Deviations,Closed,20/03/2026
696,5410025,1582.0,NaN,M60444,NaN,NaN,MXXX,MAQ-Incidente-NEUROBION 10 000 DC Lote M60444_...,5410025,"Lemus, Juan",14/04/2026,13/05/2026,Montserrat Velazquez,Closed,13/05/2026
697,NaN,NaN,NaN,NaN,NaN,NaN,MXXX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [30]:
# Apply the function to each row of BatchTracker_df and create the new column
BatchTracker_df['Deviation_Info'] = BatchTracker_df.apply(create_deviation_info, axis=1)

# Apply the function to each row of BatchTracker_df and create the new column 'Deviation Due Date'
BatchTracker_df['Deviation Due Date'] = BatchTracker_df.apply(create_deviation_due_date, axis=1)

# Display the updated BatchTracker_df
BatchTracker_df.to_excel("Deviations_BatchTracker.xlsx")
BatchTracker_df = BatchTracker_df



In [31]:

# Apply the function to each row of BatchTracker_df and create the new column
BatchTracker_df['Deviation_Info'] = BatchTracker_df.apply(create_deviation_info, axis=1)

# Apply the function to each row of BatchTracker_df and create the new column 'Deviation Due Date'
BatchTracker_df['Deviation Due Date'] = BatchTracker_df.apply(create_deviation_due_date, axis=1)

# Display the updated BatchTracker_df
BatchTracker_df.to_excel("Deviations_BatchTracker.xlsx")
BatchTracker_df = BatchTracker_df


## Concatenación Revisores Fabri

In [32]:

# Select columns to concatenate 
columns_fabri_liquidos=['LOTE', 'PRODUCTO', 'FECHA DE FIN DE PROCESO','FECHA DE ENTREGA A DICTAMEN', 'Estatus']
columns_fabri_solidos=['Lote', 'Producto', 'Fecha fin de proceso','Fecha calidad',"Comentarios"]
columns_fabri_tiroideos=['Lote ','Producto','Fin  de Proceso','Fecha Entrega a\n QA','Estatus']
columns_batch_tracker_fabri= ['LOTE DE GRANEL','DESCRIPCIÓN', 'FIN FABRI REAL', 'PR_ENTREGA EXPEDIENTE GR REAL',"PR_Estatus"]

# Rename columns for each DataFrame
fabri_liquidos_QA_piso.rename(columns={
    'LOTE': 'LOTE DE GRANEL',
    'PRODUCTO': 'DESCRIPCIÓN',
    'FECHA DE FIN DE PROCESO': 'FIN FABRI REAL',
    'FECHA DE ENTREGA A DICTAMEN': 'PR_ENTREGA EXPEDIENTE GR REAL',
    'Estatus': 'PR_Estatus'
}, inplace=True)
fabri_solidos_QA_piso.rename(columns={
    'Lote': 'LOTE DE GRANEL',
    'Producto': 'DESCRIPCIÓN',
    'Fecha fin de proceso': 'FIN FABRI REAL',
    'Fecha calidad': 'PR_ENTREGA EXPEDIENTE GR REAL',
    'Comentarios': 'PR_Estatus'
}, inplace=True)
fabri_tiroideos_QA_piso.rename(columns={
    'Lote ': 'LOTE DE GRANEL',
    'Producto': 'DESCRIPCIÓN',
    'Fin  de Proceso': 'FIN FABRI REAL',
    'Fecha Entrega a\n QA': 'PR_ENTREGA EXPEDIENTE GR REAL',
    'Estatus': 'PR_Estatus'
}, inplace=True)


# Concatenate the DataFrames
combined_Fabri_df = pd.concat([fabri_liquidos_QA_piso[columns_batch_tracker_fabri], fabri_solidos_QA_piso[columns_batch_tracker_fabri], fabri_tiroideos_QA_piso[columns_batch_tracker_fabri]], ignore_index=True)

combined_Fabri_df= combined_Fabri_df.dropna(subset=['LOTE DE GRANEL'])

# Ensure 'FIN FABRI REAL' is in datetime format
combined_Fabri_df['FIN FABRI REAL'] = pd.to_datetime(combined_Fabri_df['FIN FABRI REAL'], errors='coerce')

# Filter the DataFrame for dates in the year 2025
combined_Fabri_df = combined_Fabri_df[combined_Fabri_df['FIN FABRI REAL'] >"2024-11-01"]
combined_Fabri_df['LOTE DE GRANEL'] = combined_Fabri_df['LOTE DE GRANEL'].str.replace(' ', '', regex=True)

combined_Fabri_df.to_excel(r"C:\Users\x286384\MerckGroup\Digitalización - Documents\DigitalProjectsLibrary\Calibration\Tendencias\OTIF\ConsolidatedDB\RV_Consolidated_Fabri_Review.xlsx", index=False)




In [33]:
combined_Fabri_df

,LOTE DE GRANEL,DESCRIPCIÓN,FIN FABRI REAL,PR_ENTREGA EXPEDIENTE GR REAL,PR_Estatus
1,M53228,AMP.ALGINAC 5000 II,2025-07-18,2025-07-25 00:00:00,Con Supervisor
2,M50197,AMP.CITONEURIN I DOUBLE FILTRATION,2025-07-23,2025-07-30 00:00:00,Entregado QA
3,M50211,AMP.CITONEURIN 1000 II Double Filtration,2025-07-25,2025-07-31 00:00:00,Entregado QA
5,M53647,AMP. ALGINAC 5000 I,2025-07-17,2025-07-23 00:00:00,Entregado QA
6,M52724,AMP. DEXABIÓN 1,2025-07-01,2025-07-08 00:00:00,Entregado QA
...,...,...,...,...,...
11185,M62884,EUTHYROX NF 75 MCG LEVOTHYROXINE TABLET,2026-09-24,NaT,En Correccion
11189,M63365,EUTHYROX NF 100 - BULK (1 TAB) BRA,2026-09-24,NaT,En Revision
11190,M63364,EUTHYROX NF 100 - BULK (1 TAB) BRA,2026-09-24,NaT,En Espera Para Revision
11191,M63366,EUTHYROX NF 100 - BULK (1 TAB) BRA,2026-09-25,NaT,En Espera Para Revision


## Concatenación Revisores Acondi

In [34]:


columns_acondi = [ 'LOTE PT', 'CLAVE', 'PRODUCTO','Fecha término de proceso MES','Entrega a Calidad']
columns_batch_tracker_acondi= ['LOTE PT','SKU', 'DESCRIPCIÓN', 'ACO_FIN ACONDI REAL', 'PT_ENTREGA EXPEDIENTE REAL']

import pandas as pd


# Rename columns for each DataFrame
acondi_liquidos_QA_piso.rename(columns={
    'LOTE PT': 'LOTE PT',
    'CLAVE': 'SKU',
    'PRODUCTO': 'DESCRIPCIÓN',
    'Fecha término de proceso MES': 'ACO_FIN ACONDI REAL',
    'Entrega a Calidad': 'PT_ENTREGA EXPEDIENTE REAL'
}, inplace=True)

acondi_solidos_QA_piso.rename(columns={
    'LOTE PT': 'LOTE PT',
    'CLAVE': 'SKU',
    'PRODUCTO': 'DESCRIPCIÓN',
    'Fecha término de proceso MES': 'ACO_FIN ACONDI REAL',
    'Entrega a Calidad': 'PT_ENTREGA EXPEDIENTE REAL'
}, inplace=True)

acondi_tiroideos_QA_piso.rename(columns={
    'LOTE PT': 'LOTE PT',
    'CLAVE': 'SKU',
    'PRODUCTO': 'DESCRIPCIÓN',
    'Fecha término de proceso MES': 'ACO_FIN ACONDI REAL',
    'Entrega a Calidad': 'PT_ENTREGA EXPEDIENTE REAL'
}, inplace=True)

# Concatenate the DataFrames
combined_acondi_df = pd.concat([acondi_liquidos_QA_piso[columns_batch_tracker_acondi],
                                 acondi_solidos_QA_piso[columns_batch_tracker_acondi], 
                                 acondi_tiroideos_QA_piso[columns_batch_tracker_acondi]], ignore_index=True)

# Delete rows where 'LOTE PT' has NaN values
combined_acondi_df = combined_acondi_df.dropna(subset=['LOTE PT'])

# Ensure 'ACO_FIN ACONDI REAL' is in datetime format
combined_acondi_df['ACO_FIN ACONDI REAL'] = pd.to_datetime(combined_acondi_df['ACO_FIN ACONDI REAL'], errors='coerce')

# Optionally, filter for specific dates if needed (e.g., for the year 2025)
combined_acondi_df = combined_acondi_df[combined_acondi_df['ACO_FIN ACONDI REAL'] >"2024-11-01"]
combined_acondi_df['LOTE PT'] = combined_acondi_df['LOTE PT'].str.replace(' ', '', regex=True)

# Display the cleaned DataFrame
combined_acondi_df.to_excel(r"C:\Users\x286384\MerckGroup\Digitalización - Documents\DigitalProjectsLibrary\Calibration\Tendencias\OTIF\ConsolidatedDB\RV_Consolidated_Acondi_Review.xlsx", index=False)
combined_acondi_df
# If filtering was applied, uncomment the line below to display the filtered DataFrame
# print(filtered_acondi_df)



,LOTE PT,SKU,DESCRIPCIÓN,ACO_FIN ACONDI REAL,PT_ENTREGA EXPEDIENTE REAL
1357,M44403,MX11378,ALGINAC 5000 BOX W/6 AMP SOL (1ML/2ML),2024-11-04 15:48:00,2024-11-06
1359,M42792,MX11359,CITONEURIN 5000 BOX W/6 AMP SOL (1ML),2024-11-05 11:59:00,2024-11-06
1360,M44369,3.47369.0001,Neurobion 25000DC-(1) GTM,2024-11-03 08:49:00,2024-11-04
1363,M43552,MX11000,CISTICID 600 MG TAB - (25) MEX,2024-11-06 12:59:00,2024-11-07
1364,M43553,MX11000,CISTICID 600 MG TAB - (25) MEX,2024-11-14 10:25:00,2024-11-19
...,...,...,...,...,...
13015,M62808,3045740042,LEVOTIROXINA NF 100 MCG TAB-(100) MEX Oct L...,2026-09-23 13:15:00,NaT
13016,M62809,3045740042,LEVOTIROXINA NF 100 MCG TAB-(100) MEX Oct L...,2026-09-23 23:26:00,NaT
13017,M62816,3045740042,LEVOTIROXINA NF 100 MCG TAB-(100) MEX Oct L...,2026-09-24 08:10:00,NaT
13018,M62806,3045740042,LEVOTIROXINA NF 100 MCG TAB-(100) MEX Oct L...,2026-09-24 18:02:00,NaT


## Concatenación QC

In [35]:

columns_qc_general   = ['LOTE ','PRODUCTO O FAMILIA ','FECHA DE ENTRADA AL LABORATORIO', 'ENTREGA A DICTAMEN\n(mes/dia/año', 'ESTATUS\npara QC']
columns_qc_tiroideos = [ 'LOTE ', 'PRODUCTO(Nombre de plan de muestreo)', 'FECHA DE ENTRADA AL LABORATORIO','ENTREGA A DICTAMEN\n(mes/dia/año', 'ESTATUS\npara QC' ]

columns_batch_tracker_QC= [ 'LOTE' ,'DESCRIPCIÓN' ,'PR_ENTREGA MUESTRA REAL',  'ENTREGA ÓRDEN ANALÍTICA REAL', "QC_Estatus" ]

# Rename columns for each DataFrame
QC_general.rename(columns={
    'LOTE ': 'LOTE',
    'PRODUCTO O FAMILIA ': 'DESCRIPCIÓN',
    'FECHA DE ENTRADA AL LABORATORIO': 'PR_ENTREGA MUESTRA REAL',
    'ENTREGA A DICTAMEN\n(mes/dia/año': 'ENTREGA ÓRDEN ANALÍTICA REAL',
    'ESTATUS\npara QC': 'QC_Estatus'
}, inplace=True)

QC_tiroides.rename(columns={
    'LOTE ': 'LOTE',
    'PRODUCTO(Nombre de plan de muestreo)': 'DESCRIPCIÓN',
    'FECHA DE ENTRADA AL LABORATORIO': 'PR_ENTREGA MUESTRA REAL',
    'ENTREGA A DICTAMEN\n(mes/dia/año': 'ENTREGA ÓRDEN ANALÍTICA REAL',
    'ESTATUS\npara QC': 'QC_Estatus'
}, inplace=True)

# Concatenate the DataFrames
combined_qc_df = pd.concat([QC_general[columns_batch_tracker_QC], QC_tiroides[columns_batch_tracker_QC]], ignore_index=True)

# Delete rows where 'LOTE' has NaN values
combined_qc_df = combined_qc_df.dropna(subset=['LOTE'])

# Ensure 'PR_ENTREGA MUESTRA REAL' is in datetime format
combined_qc_df['PR_ENTREGA MUESTRA REAL'] = pd.to_datetime(combined_qc_df['PR_ENTREGA MUESTRA REAL'], errors='coerce')
combined_qc_df['ENTREGA ÓRDEN ANALÍTICA REAL'] = pd.to_datetime(combined_qc_df['ENTREGA ÓRDEN ANALÍTICA REAL'], errors='coerce')
# Filter the DataFrame for dates in the year 2025
combined_qc_df = combined_qc_df[combined_qc_df['PR_ENTREGA MUESTRA REAL'] >"2024-11-01"]
combined_qc_df['LOTE'] = combined_qc_df['LOTE'].str.replace(' ', '', regex=True)


# Display the filtered DataFrame
combined_qc_df.to_excel(r"C:\Users\x286384\MerckGroup\Digitalización - Documents\DigitalProjectsLibrary\Calibration\Tendencias\OTIF\ConsolidatedDB\QC_Consolidated_QC.xlsx", index=False)

In [36]:
combined_qc_df

,LOTE,DESCRIPCIÓN,PR_ENTREGA MUESTRA REAL,ENTREGA ÓRDEN ANALÍTICA REAL,QC_Estatus
5582,M43409,Dexa-Citoneurin PT,2024-11-04,2024-11-12,ENTREGADO
5583,M44403,Alginac PT,2024-11-04,2024-11-12,ENTREGADO
5584,M43417,Neurobion PT,2024-11-04,2024-11-12,ENTREGADO
5585,M42791,Citoneurin PT,2024-11-04,2024-11-12,ENTREGADO
5586,M43506,Amp.Citoneurin I,2024-11-06,2024-11-21,ENTREGADO
...,...,...,...,...,...
17302,M62880,Euthyrox NF 75,2026-09-24,NaT,POR ANALIZAR
17303,M62882,Euthyrox NF 75,2026-09-24,NaT,POR ANALIZAR
17304,M62979,Euthyrox NF 88,2026-09-24,NaT,POR ANALIZAR
17305,M63362,Euthyrox NF 100,2026-09-24,NaT,POR ANALIZAR


## Concatenación Planeación

### Tiroideos

In [37]:

import xlsxwriter
print(xlsxwriter.__version__)
columns_tiroideos_fabri_NF=['Clave', 'Producto', 'Lote','Compresión',  'Muestras',  'Entrega de orden a QA','LIBERACION ANÁLISIS','Dictámen', 'CANTIDAD', 'Orden', 'TAB' ]
plan_tiroideos_fabri_NF.keys()

columns_tiroideos_fabri_VF=['Clave', 'Producto', 'Lote','Compresión',  'Muestras',  'Entrega de orden a QA','LIBERACION ANÁLISIS','Dictámen', 'CANTIDAD', 'Orden', 'TAB' ]
plan_tiroideos_fabri_VF.keys()


df_tiroideos_fabri_VF = clean_rows_based_on_prefix(plan_tiroideos_fabri_VF[columns_tiroideos_fabri_VF], 'Lote')
df_tiroideos_fabri_NF = clean_rows_based_on_prefix(plan_tiroideos_fabri_NF[columns_tiroideos_fabri_NF], 'Lote')


# Stack the data frames
df_fabri_tiroideos_consolidated = pd.concat([df_tiroideos_fabri_VF, df_tiroideos_fabri_NF], ignore_index=True)

# Create a mapping for renaming
rename_mapping = {
    'Clave': 'SKU',
    'Producto': 'DESCRIPCIÓN',
    'Lote': 'LOTE DE GRANEL',
    'Compresión': 'FIN FABRI PLAN',
    'Muestras': 'PR_ENTREGA MUESTRA PLAN',
    'Entrega de orden a QA': 'PR_ENTREGA EXPEDIENTE GR PLAN',
    'LIBERACION ANÁLISIS': 'ENTREGA ÓRDEN ANALÍTICA PLAN',
    'CANTIDAD': 'GR_CANTIDAD',
    "Orden": "ORDEN GR",
    "TAB": "RECURSO GR"
}

# Rename the columns
df_fabri_tiroideos_consolidated.rename(columns=rename_mapping, inplace=True)
df_fabri_tiroideos_consolidated["PR_ENTREGA EXPEDIENTE GR PLAN"]= pd.to_datetime(df_fabri_tiroideos_consolidated["FIN FABRI PLAN"], errors="coerce")+ timedelta(days=7)
df_fabri_tiroideos_consolidated["GR_CANTIDAD C/YIELD"]= df_fabri_tiroideos_consolidated[["GR_CANTIDAD"]]

df_fabri_tiroideos_consolidated["LOTE DE GRANEL"]=df_fabri_tiroideos_consolidated["LOTE DE GRANEL"].apply(lambda x: x.strip())
df_fabri_tiroideos_consolidated.to_excel(r"C:\Users\x286384\MerckGroup\Digitalización - Documents\DigitalProjectsLibrary\Calibration\Tendencias\OTIF\ConsolidatedDB\PL_Fabri_Tiroideos_Consolidated.xlsx", index=False)
print(df_fabri_tiroideos_consolidated.keys())
# Save DataFrame to Excel file with table formatting
with pd.ExcelWriter(r"C:\Users\{0}\MerckGroup\Seguimiento Servicio GMS P&G-BP - TrackWise_Deviations\PL_Fabri_Tiroideos_Consolidated.xlsx".format(muid), engine='xlsxwriter') as writer:
    df_fabri_tiroideos_consolidated.to_excel(writer, index=False, sheet_name='BT_Fabri_Tiroides_consolidates')
    
    # Get the xlsxwriter workbook and worksheet objects
    workbook  = writer.book
    worksheet = writer.sheets['BT_Fabri_Tiroides_consolidates']
    
    # Create a table
    num_rows, num_cols = df_fabri_tiroideos_consolidated.shape
    worksheet.add_table(0, 0, num_rows, num_cols - 1, {'name': 'Fabri_Tiroides', 'columns': [{'header': col} for col in df_fabri_tiroideos_consolidated.columns]})







3.2.9
Index(['SKU', 'DESCRIPCIÓN', 'LOTE DE GRANEL', 'FIN FABRI PLAN',
       'PR_ENTREGA MUESTRA PLAN', 'PR_ENTREGA EXPEDIENTE GR PLAN',
       'ENTREGA ÓRDEN ANALÍTICA PLAN', 'Dictámen', 'GR_CANTIDAD', 'ORDEN GR',
       'RECURSO GR', 'GR_CANTIDAD C/YIELD'],
      dtype='str')


In [38]:
columns_acondi_tiroideos_L10=['CLAVE', 'PRODUCTO',  'LOTE PT', 'LOTE GRANEL', 'Comentarios',  'FIN', 'LIBERACIÓN', 'Fecha de requerimiento','OTIF', 'CANTIDAD', 'Recurso', 'ORDEN']
plan_tiroideos_acondi_L10.keys()
columns_acondi_tiroideos_L9=['CLAVE', 'PRODUCTO',  'LOTE PT', 'LOTE GRANEL', 'Comentarios',  'FIN', 'LIBERACIÓN', 'Fecha de requerimiento','OTIF', 'CANTIDAD', 'Recurso', 'ORDEN']
plan_tiroideos_acondi_L9.keys()
columns_acondi_tiroideos_L8=['CLAVE', 'PRODUCTO',  'LOTE PT', 'LOTE GRANEL', 'Comentarios',  'FIN', 'LIBERACIÓN', 'Fecha de requerimiento','OTIF', 'CANTIDAD', 'Recurso', 'ORDEN']
plan_tiroideos_acondi_L8.keys()
columns_acondi_tiroideos_LM=['CLAVE', 'PRODUCTO',  'LOTE PT', 'LOTE GRANEL', 'Comentarios',  'Entrega', 'Fecha de requerimiento','LIBERACIÓN','OTIF', 'CANTIDAD','ORDEN']
plan_tiroideos_acondi_LM.keys()


df_tiroideos_acondi_L10 = clean_rows_based_on_prefix(plan_tiroideos_acondi_L10[columns_acondi_tiroideos_L10], 'LOTE PT')
df_tiroideos_acondi_L9  = clean_rows_based_on_prefix(plan_tiroideos_acondi_L9[columns_acondi_tiroideos_L9], 'LOTE PT')
df_tiroideos_acondi_L8  = clean_rows_based_on_prefix(plan_tiroideos_acondi_L8[columns_acondi_tiroideos_L8], 'LOTE PT')
df_tiroideos_acondi_LM  = clean_rows_based_on_prefix(plan_tiroideos_acondi_LM[columns_acondi_tiroideos_LM], 'LOTE PT')
df_tiroideos_acondi_LM["Recurso"]="MAQUILA"

# Stack the data frames
df_acondi_tiroideos_consolidated = pd.concat([df_tiroideos_acondi_L10, df_tiroideos_acondi_L9, df_tiroideos_acondi_L8], ignore_index=True)

# Create a mapping for renaming
rename_mapping = {
    'CLAVE': 'SKU',
    'PRODUCTO': 'DESCRIPCIÓN',
    'LOTE GRANEL': 'LOTE DE GRANEL',
    'LOTE PT': 'LOTE PT',
    'Comentarios': 'PL_STATUS',
    'FIN': 'ACO_FIN ACONDI PLAN',
    'LIBERACIÓN': 'FECHA LIBERACION V0',
    'Fecha de requerimiento': 'FECHA REQ',
    'OTIF':'FECHA OTIF', 
    'CANTIDAD': 'PT_CANTIDAD',
    "Recurso": "RECURSO PT",
    "ORDEN": "ORDEN PT"

}

rename_mapping_maquila = {
    'CLAVE': 'SKU',
    'PRODUCTO': 'DESCRIPCIÓN',
    'LOTE GRANEL': 'LOTE DE GRANEL',
    'LOTE PT': 'LOTE PT',
    'Comentarios': 'PL_STATUS',
    'Entrega': 'ACO_FIN ACONDI PLAN',
    'LIBERACIÓN': 'FECHA LIBERACION V0',
    'Fecha de requerimiento': 'FECHA REQ',
    'OTIF':'FECHA OTIF', 
    'CANTIDAD': 'PT_CANTIDAD' ,
    "Recurso": "RECURSO PT",
    "ORDEN": "ORDEN PT"
}


# Rename the columns
df_acondi_tiroideos_consolidated.rename(columns=rename_mapping, inplace=True)
df_tiroideos_acondi_LM.rename(columns=rename_mapping_maquila, inplace=True)

df_acondi_tiroideos_consolidated["PT_ENTREGA EXPEDIENTE PLAN"]= pd.to_datetime(df_acondi_tiroideos_consolidated["ACO_FIN ACONDI PLAN"], errors="coerce")+ timedelta(days=4)

final_columns_acondi_tiroideos = ['SKU', 'DESCRIPCIÓN', 'LOTE PT', 'LOTE DE GRANEL', 'PL_STATUS',
 'ACO_FIN ACONDI PLAN', 'FECHA LIBERACION V0', 'FECHA REQ', 'FECHA OTIF', 'PT_ENTREGA EXPEDIENTE PLAN', 'PT_CANTIDAD' , "RECURSO PT", "ORDEN PT"]

# Merge the two DataFrames on 'LOTE PT'
# Mergin dataframes to substitute dates for MAQUILA, new note: revisar compatibilidad con linea manual
df_acondi_tiroideos_consolidated_final = df_acondi_tiroideos_consolidated[final_columns_acondi_tiroideos].merge(
    df_tiroideos_acondi_LM[['LOTE PT', 'ACO_FIN ACONDI PLAN', 'FECHA LIBERACION V0', 'FECHA REQ', 'FECHA OTIF']],
    on='LOTE PT',
    how='left',
    suffixes=('', '_y')  # This will help avoid confusion with column names
)

# Update the values in the original DataFrame based on the merged DataFrame
df_acondi_tiroideos_consolidated_final['ACO_FIN ACONDI PLAN'] = df_acondi_tiroideos_consolidated_final['ACO_FIN ACONDI PLAN_y'].combine_first(df_acondi_tiroideos_consolidated_final['ACO_FIN ACONDI PLAN'])
df_acondi_tiroideos_consolidated_final['FECHA LIBERACION V0'] = df_acondi_tiroideos_consolidated_final['FECHA LIBERACION V0_y'].combine_first(df_acondi_tiroideos_consolidated_final['FECHA LIBERACION V0'])
df_acondi_tiroideos_consolidated_final['FECHA REQ']           = pd.to_datetime(df_acondi_tiroideos_consolidated_final['FECHA REQ_y'], errors='coerce').combine_first( pd.to_datetime(df_acondi_tiroideos_consolidated_final['FECHA REQ'], errors='coerce'))
df_acondi_tiroideos_consolidated_final['FECHA OTIF']          = pd.to_datetime(df_acondi_tiroideos_consolidated_final['FECHA OTIF_y'], errors='coerce').combine_first( pd.to_datetime(df_acondi_tiroideos_consolidated_final['FECHA OTIF'], errors='coerce'))
df_acondi_tiroideos_consolidated_final['TYPE_BATCH']          = "E2E"

df_acondi_tiroideos_consolidated_final["PT_CANTIDAD C/YIELD"]= df_acondi_tiroideos_consolidated_final["PT_CANTIDAD"]
# Save the updated DataFrame to Excel
df_acondi_tiroideos_consolidated_final["LOTE PT"]=df_acondi_tiroideos_consolidated_final["LOTE PT"].apply(lambda x: x.strip())
df_acondi_tiroideos_consolidated_final["LOTE DE GRANEL"]=df_acondi_tiroideos_consolidated_final["LOTE DE GRANEL"].apply(lambda x: str(x).strip())

df_acondi_tiroideos_consolidated_final.to_excel(r"C:\Users\x286384\MerckGroup\Digitalización - Documents\DigitalProjectsLibrary\Calibration\Tendencias\OTIF\ConsolidatedDB\PL_Acondi_Tiroideos_Consolidated.xlsx", index=False)

# Save DataFrame to Excel file with table formatting
# Save DataFrame to Excel file with table formatting
with pd.ExcelWriter(r"C:\Users\{0}\MerckGroup\Seguimiento Servicio GMS P&G-BP - TrackWise_Deviations\PL_Acondi_Tiroideos_Consolidated_1.xlsx".format(muid), engine='xlsxwriter') as writer:
    df_acondi_tiroideos_consolidated_final.to_excel(writer, index=False, sheet_name='PL_Acondi_Tiroideos')
    
    # Get the xlsxwriter workbook and worksheet objects
    workbook  = writer.book
    worksheet = writer.sheets['PL_Acondi_Tiroideos']
    
    # Create a table
    num_rows, num_cols = df_acondi_tiroideos_consolidated_final.shape
    worksheet.add_table(0, 0, num_rows, num_cols - 1, {'name': 'Acondi_Tiroides', 'columns': [{'header': col} for col in df_acondi_tiroideos_consolidated_final.columns]})





C:\Users\x286384\AppData\Local\Temp\ipykernel_36140\506577249.py:72: Pandas4Warning: Silently casting non-datetime 'other' to datetime in Series.combine_first is deprecated and will be removed in a future version. Explicitly cast before calling combine_first instead.
  df_acondi_tiroideos_consolidated_final['ACO_FIN ACONDI PLAN'] = df_acondi_tiroideos_consolidated_final['ACO_FIN ACONDI PLAN_y'].combine_first(df_acondi_tiroideos_consolidated_final['ACO_FIN ACONDI PLAN'])


### Solidos

In [39]:
columns_fabri_solidos_PLV=['Clave', 'Producto', 'Lote','Mezclado','ENTREGA MUESTRA',  'FECHA DE DICTAMEN']
plan_solidos_fabri_TAB12.keys()

Index(['Unnamed: 0', 'Recurso', 'Clave', 'Producto', 'Lote', 'Orden',
       'CANTIDAD', 'CANTIDAD C/YIELD', 'Pesaje', 'Real',
       'Granulación /Humectacion', 'Real.1', 'Mezclado', 'Real.2',
       'Compresión', 'Real.3', 'Recubrimiento', 'Real.4', 'Envasado',
       'Revision', 'fecha de cierre', 'ENTREGA MUESTRA', 'ENTREGA ORDEN',
       'DÍAS DE ANÁLISIS', 'LIBERACIÓN ANÁLISIS', 'UD', 'Unnamed: 26',
       'Unnamed: 27', 'Unnamed: 28', '% YIELD', 'ABSORCION'],
      dtype='str')

In [40]:
columns_fabri_solidos_TAB12=['Clave', 'Producto', 'Lote','Recubrimiento','ENTREGA MUESTRA',  'LIBERACIÓN ANÁLISIS', 'UD', "CANTIDAD", "CANTIDAD C/YIELD", "Recurso", "Orden"]
plan_solidos_fabri_TAB12.keys()
columns_fabri_solidos_TAB13=['Clave', 'Producto', 'Lote','Recubrimiento','Compresión','ENTREGA MUESTRA',  'LIBERACIÓN ANÁLISIS', 'UD',  "CANTIDAD", "CANTIDAD C/YIELD", "Recurso", "Orden"]
plan_solidos_fabri_TAB13.keys()
columns_fabri_solidos_TAB19=['Clave', 'Producto', 'Lote','Recub','Compresión','ENTREGA MUESTRA',  'LIBERACIÓN ANÁLISIS' , 'UD', "CANTIDAD", "CANTIDAD C/YIELD", "Recurso", "Orden"]
plan_solidos_fabri_TAB19.keys()
columns_fabri_solidos_TAB24=['Clave', 'Producto', 'Lote','Compresión','ENTREGA MUESTRA',  'LIBERACIÓN ANÁLISIS', 'UD', "CANTIDAD", "CANTIDAD C/YIELD", "Recurso", "Orden"]
plan_solidos_fabri_TAB24.keys()
columns_fabri_solidos_ENV=['Clave', 'Producto', 'Lote', 'Lote Granel','Envasado','FECHA DE DICTAMEN', 'UD',  "CANTIDAD", "CANTIDAD C/YIELD", "Orden"]
plan_solidos_fabri_ENV.keys()
columns_fabri_solidos_PLV=['Clave', 'Producto', 'Lote','Mezclado','ENTREGA MUESTRA','FECHA DE DICTAMEN', 'UD', "CANTIDAD", "Recurso", "Orden"]
plan_solidos_fabri_PLV.keys()

columns_fabri_solidos_BT=['Clave', 'Producto', 'Lote','FIN','ENTREGA MUESTRA',  'LIBERACIÓN ANÁLISIS', "UD",  "CANTIDAD", "CANTIDAD C/YIELD","Recurso", "Orden"]

df_plan_solidos_fabri_TAB12 = clean_rows_based_on_prefix(plan_solidos_fabri_TAB12[columns_fabri_solidos_TAB12], 'Lote')
df_plan_solidos_fabri_TAB13 = clean_rows_based_on_prefix(plan_solidos_fabri_TAB13[columns_fabri_solidos_TAB13], 'Lote')
df_plan_solidos_fabri_TAB19 = clean_rows_based_on_prefix(plan_solidos_fabri_TAB19[columns_fabri_solidos_TAB19], 'Lote')
df_plan_solidos_fabri_TAB24 = clean_rows_based_on_prefix(plan_solidos_fabri_TAB24[columns_fabri_solidos_TAB24], 'Lote')
df_plan_solidos_fabri_ENV   = clean_rows_based_on_prefix(plan_solidos_fabri_ENV[columns_fabri_solidos_ENV], 'Lote')
df_plan_solidos_fabri_PLV   = clean_rows_based_on_prefix(plan_solidos_fabri_PLV[columns_fabri_solidos_PLV], 'Lote')

# For TAB13, add the FIN column comparing Recubrimiento and Compresion
df_plan_solidos_fabri_ENV["Recurso"] = "ENVASADO"
# Calculate the maximum and convert to datetime for TAB13
df_plan_solidos_fabri_TAB13= set_fin_column(df=df_plan_solidos_fabri_TAB13, column_name_1="Recubrimiento",column_name_2="Compresión")
df_plan_solidos_fabri_TAB19= set_fin_column(df=df_plan_solidos_fabri_TAB19, column_name_1="Recub",column_name_2="Compresión")



# Rename columns for TAB12 and TAB24
df_plan_solidos_fabri_TAB12.rename(columns={'Recubrimiento': 'FIN'}, inplace=True)
df_plan_solidos_fabri_TAB24.rename(columns={'Compresión': 'FIN'}, inplace=True)
df_plan_solidos_fabri_PLV.rename(columns={'Mezclado': 'FIN',
                                           'FECHA DE DICTAMEN': 'LIBERACIÓN ANÁLISIS'}, inplace=True)
df_plan_solidos_fabri_PLV["CANTIDAD C/YIELD"]=df_plan_solidos_fabri_PLV[["CANTIDAD"]]
# Stack the data frames
df_fabri_solidos_consolidated = pd.concat([df_plan_solidos_fabri_TAB12[columns_fabri_solidos_BT], 
                                           df_plan_solidos_fabri_TAB13[columns_fabri_solidos_BT], 
                                           df_plan_solidos_fabri_TAB19[columns_fabri_solidos_BT],
                                           df_plan_solidos_fabri_TAB24[columns_fabri_solidos_BT],
                                           df_plan_solidos_fabri_PLV[columns_fabri_solidos_BT]], 
                                           ignore_index=True)

# Create a mapping for renaming
rename_mapping = {
    'Clave': 'SKU',
    'Producto': 'DESCRIPCIÓN',
    'Lote': 'LOTE DE GRANEL',
    'FIN': 'FIN FABRI PLAN',
    'ENTREGA MUESTRA': 'PR_ENTREGA MUESTRA PLAN',
    'LIBERACIÓN ANÁLISIS': 'ENTREGA ÓRDEN ANALÍTICA PLAN',
    'CANTIDAD': 'GR_CANTIDAD',
    'CANTIDAD C/YIELD': 'GR_CANTIDAD C/YIELD',
    'Recurso': "RECURSO GR",
    'Orden':"ORDEN GR"
}

# Rename the columns
df_fabri_solidos_consolidated.rename(columns=rename_mapping, inplace=True)
df_fabri_solidos_consolidated["PR_ENTREGA EXPEDIENTE GR PLAN"]= pd.to_datetime(df_fabri_solidos_consolidated["FIN FABRI PLAN"], errors="coerce")+ timedelta(days=7)
df_fabri_solidos_consolidated["LOTE DE GRANEL"]=df_fabri_solidos_consolidated["LOTE DE GRANEL"].apply(lambda x: x.strip())
df_fabri_solidos_consolidated.to_excel(r"C:\Users\x286384\MerckGroup\Digitalización - Documents\DigitalProjectsLibrary\Calibration\Tendencias\OTIF\ConsolidatedDB\PL_Fabri_Solidos_Consolidated.xlsx", index=False)

with pd.ExcelWriter(r"C:\Users\{0}\MerckGroup\Seguimiento Servicio GMS P&G-BP - TrackWise_Deviations\PL_Fabri_Solidos_Consolidated.xlsx".format(muid), engine='xlsxwriter') as writer:
    df_fabri_solidos_consolidated.to_excel(writer, index=False, sheet_name='PL_Fabri_Solidos_Consolidated')
    
    # Get the xlsxwriter workbook and worksheet objects
    workbook  = writer.book
    worksheet = writer.sheets['PL_Fabri_Solidos_Consolidated']
    
    # Create a table
    num_rows, num_cols = df_fabri_solidos_consolidated.shape
    worksheet.add_table(0, 0, num_rows, num_cols - 1, {'name': 'Fabri_Solidos_Consolidated', 'columns': [{'header': col} for col in df_fabri_solidos_consolidated.columns]})





In [41]:
columns_solidos_acondi_L3=['CLAVE', 'DESCP', 'ORDEN', 'LOTE PT','LOTE GRANEL','COMENTARIOS','FIN', 'FECHA REQ CLIENTE','OTIF +14', 'FECHA DICTAMEN',  "CANTIDAD", "CANTIDAD C/YIELD", "LINEA", "ORDEN"]
plan_solidos_acondi_L3.keys()
columns_solidos_acondi_L4=['CLAVE', 'DESCP', 'ORDEN', 'LOTE PT','LOTE GRANEL','COMENTARIOS','FIN', 'FECHA REQ CLIENTE','OTIF +14', 'FECHA DICTAMEN',  "CANTIDAD", "CANTIDAD C/YIELD", "LINEA", "ORDEN"]
plan_solidos_acondi_L4.keys()
columns_solidos_acondi_L5=['CLAVE', 'DESCP', 'ORDEN', 'LOTE PT','LOTE GRANEL','COMENTARIOS','FIN', 'FECHA REQ CLIENTE','OTIF +14', 'FECHA DICTAMEN',  "CANTIDAD", "CANTIDAD C/YIELD", "LINEA", "ORDEN"]
plan_solidos_acondi_L5.keys()
columns_solidos_acondi_L6=['CLAVE', 'DESCP', 'ORDEN', 'LOTE PT','LOTE GRANEL','COMENTARIOS','FIN', 'FECHA REQ CLIENTE','OTIF +14', 'FECHA DICTAMEN',  "CANTIDAD", "CANTIDAD C/YIELD", "LINEA", "ORDEN"]
plan_solidos_acondi_L6.keys()
columns_solidos_acondi_L7=['CLAVE', 'DESCP', 'ORDEN', 'LOTE PT','LOTE GRANEL','COMENTARIOS','FIN', 'FECHA REQ CLIENTE','OTIF +14', 'FECHA DICTAMEN',  "CANTIDAD", "CANTIDAD C/YIELD", "LINEA", "ORDEN"]
plan_solidos_acondi_L7.keys()
columns_solidos_acondi_LC=['CLAVE', 'DESCP', 'ORDEN', 'LOTE PT','LOTE GRANEL','COMENTARIOS','FIN', 'FECHA REQ CLIENTE','OTIF +14', 'FECHA DICTAMEN',  "CANTIDAD", "CANTIDAD C/YIELD", "LINEA", "ORDEN"]
plan_solidos_acondi_LC.keys()
columns_solidos_acondi_LM=['CLAVE', 'DESCP', 'ORDEN', 'LOTE PT','LOTE GRANEL','COMENTARIOS','FIN', 'FECHA REQ CLIENTE','OTIF +14', 'FECHA DICTAMEN', "CANTIDAD", "CANTIDAD C/YIELD", "LINEA", "ORDEN"]
plan_solidos_acondi_LM.keys()


columns_acondi_solidos_BT=['CLAVE', 'DESCP', 'LOTE PT','LOTE GRANEL','COMENTARIOS','FIN', 'FECHA REQ CLIENTE','OTIF +14', 'FECHA DICTAMEN', "CANTIDAD", "CANTIDAD C/YIELD", 'ORDEN', 'LINEA']
columns_acondi_solidos_BT_PLV=['CLAVE', 'DESCP', 'LOTE PT','LOTE GRANEL','COMENTARIOS','FIN', 'FECHA REQ CLIENTE','OTIF +14', 'FECHA DICTAMEN', "CANTIDAD", "CANTIDAD C/YIELD", 'ORDEN', 'ORDEN','LINEA']

columns_fabri_solidos_PLV=['Clave', 'Producto', 'Lote','Mezclado','ENTREGA MUESTRA','FECHA DE DICTAMEN']


#df_plan_solidos_fabri_PLV["LOTE PT"]           =df_plan_solidos_fabri_PLV[['LOTE PT']]
df_plan_solidos_fabri_PLV['FECHA REQ CLIENTE'] =df_plan_solidos_fabri_PLV[['LIBERACIÓN ANÁLISIS']]
df_plan_solidos_fabri_PLV['OTIF +14']          =df_plan_solidos_fabri_PLV[['LIBERACIÓN ANÁLISIS']]
df_plan_solidos_fabri_PLV['FECHA DICTAMEN']    =df_plan_solidos_fabri_PLV[['LIBERACIÓN ANÁLISIS']]
df_plan_solidos_fabri_PLV['COMENTARIOS']    = ""
df_plan_solidos_fabri_PLV.rename(columns={'Lote': 'LOTE GRANEL',
                                          'Clave': 'CLAVE', 
                                          'Producto': "DESCP",
                                           "Recurso": "LINEA",
                                            "Orden": "ORDEN" }, inplace=True)

# Create a mapping for renaming
rename_mapping = {
    'Clave': 'CLAVE',
    'Producto': 'DESCP',
    'Lote': 'LOTE PT',
    'FIN': 'FIN FABRI PLAN',
    'ENTREGA MUESTRA': 'PR_ENTREGA MUESTRA PLAN',
    'LIBERACIÓN ANÁLISIS': 'ENTREGA ÓRDEN ANALÍTICA PLAN'
}

# Rename the columns
df_fabri_solidos_consolidated.rename(columns=rename_mapping, inplace=True)


df_plan_solidos_acondi_L3 = clean_rows_based_on_prefix(plan_solidos_acondi_L3[columns_solidos_acondi_L3], 'LOTE PT')
df_plan_solidos_acondi_L4 = clean_rows_based_on_prefix(plan_solidos_acondi_L4[columns_solidos_acondi_L4], 'LOTE PT')
df_plan_solidos_acondi_L5 = clean_rows_based_on_prefix(plan_solidos_acondi_L5[columns_solidos_acondi_L5], 'LOTE PT')
df_plan_solidos_acondi_L6 = clean_rows_based_on_prefix(plan_solidos_acondi_L6[columns_solidos_acondi_L6], 'LOTE PT')
df_plan_solidos_acondi_L7 = clean_rows_based_on_prefix(plan_solidos_acondi_L7[columns_solidos_acondi_L7], 'LOTE PT')
df_plan_solidos_acondi_LC = clean_rows_based_on_prefix(plan_solidos_acondi_LC[columns_solidos_acondi_LC], 'LOTE PT')
df_plan_solidos_acondi_LM = clean_rows_based_on_prefix(plan_solidos_acondi_LM[columns_solidos_acondi_LM], 'LOTE PT')


df_plan_solidos_acondi_LM_dolo    = df_plan_solidos_acondi_LM[df_plan_solidos_acondi_LM["CLAVE"]=="MX12167"]
print(df_plan_solidos_acondi_L3.columns)

# Rename columns for TAB12 and TAB24, checar cambio de nombre para FIN just in case
df_plan_solidos_acondi_LM.rename(columns={'RETORNO': 'FIN', "OTIF PG +14": "OTIF +14"}, inplace=True)
df_plan_solidos_acondi_LM_dolo.rename(columns={'RETORNO': 'FIN', "OTIF PG +14": "OTIF +14"}, inplace=True)


#print(len(df_plan_solidos_fabri_PLV["LOTE PT"]) + len(df_plan_solidos_acondi_L3["LOTE PT"]) + len(df_plan_solidos_acondi_L4["LOTE PT"])
#      + len(df_plan_solidos_acondi_L5["LOTE PT"])+ + len(df_plan_solidos_acondi_L6["LOTE PT"])+  len(df_plan_solidos_acondi_L7["LOTE PT"])+ 
#        len(df_plan_solidos_acondi_LC["LOTE PT"])
#      + len(df_plan_solidos_acondi_LM_dolo["LOTE PT"]))
# Stack the data frames
df_acondi_solidos_consolidated = pd.concat([df_plan_solidos_acondi_L3[columns_acondi_solidos_BT], 
                                           df_plan_solidos_acondi_L4[columns_acondi_solidos_BT], 
                                           df_plan_solidos_acondi_L5[columns_acondi_solidos_BT],
                                           df_plan_solidos_acondi_L6[columns_acondi_solidos_BT],
                                           df_plan_solidos_acondi_L7[columns_acondi_solidos_BT], 
                                           df_plan_solidos_acondi_LC[columns_acondi_solidos_BT],
                                           df_plan_solidos_acondi_LM_dolo[columns_acondi_solidos_BT]], ignore_index=True)
                                           #df_plan_solidos_fabri_PLV[columns_acondi_solidos_BT_PLV]
print(len(df_acondi_solidos_consolidated["LOTE PT"]))
# Create a mapping for renaming

rename_mapping = {
    'CLAVE': 'SKU',
    'DESCP': 'DESCRIPCIÓN',
    'LOTE GRANEL': 'LOTE DE GRANEL',
    'LOTE PT': "LOTE PT",
    'COMENTARIOS': 'PL_STATUS',
    'FIN': 'ACO_FIN ACONDI PLAN',
    'FECHA DICTAMEN': 'FECHA LIBERACION V0',
    'FECHA REQ CLIENTE': 'FECHA REQ',
    'OTIF +14':'FECHA OTIF' ,
    'CANTIDAD': 'PT_CANTIDAD',
    'CANTIDAD C/YIELD': 'PT_CANTIDAD C/YIELD',
    "LINEA": "RECURSO PT",
    "ORDEN": "ORDEN PT"
}

rename_mapping_maquila_sol = {
    'CLAVE': 'SKU',
    'DESCP': 'DESCRIPCIÓN',
    'LOTE GRANEL': 'LOTE DE GRANEL',
    'LOTE PT': "LOTE PT",
    'COMENTARIOS': 'PL_STATUS',
    'FIN': 'ACO_FIN ACONDI PLAN',
    'FECHA DICTAMEN': 'FECHA LIBERACION V0',
    'FECHA REQ CLIENTE': 'FECHA REQ',
    'OTIF +14':'FECHA OTIF' ,
    'CANTIDAD': 'PT_CANTIDAD',
    'CANTIDAD C/YIELD': 'PT_CANTIDAD C/YIELD',
    "LINEA": "RECURSO PT",
    "ORDEN": "ORDEN PT"}
# Rename the columns
df_acondi_solidos_consolidated.rename(columns=rename_mapping, inplace=True)
df_plan_solidos_acondi_LM.rename(columns=rename_mapping_maquila_sol, inplace=True)


df_acondi_solidos_consolidated["PT_ENTREGA EXPEDIENTE PLAN"]= pd.to_datetime(df_acondi_solidos_consolidated["ACO_FIN ACONDI PLAN"], errors="coerce")+ timedelta(days=4)


df_acondi_solidos_consolidated_final = df_acondi_solidos_consolidated.merge(
    df_plan_solidos_acondi_LM[['LOTE PT', 'ACO_FIN ACONDI PLAN', 'FECHA LIBERACION V0', 'FECHA REQ', 'FECHA OTIF']],
    on='LOTE PT',
    how='left',
    suffixes=('', '_y')  # This will help avoid confusion with column names
)

# Update the values in the original DataFrame based on the merged DataFrame
df_acondi_solidos_consolidated_final['ACO_FIN ACONDI PLAN'] = pd.to_datetime(df_acondi_solidos_consolidated_final['ACO_FIN ACONDI PLAN_y'], errors='coerce').combine_first(pd.to_datetime(df_acondi_solidos_consolidated_final['ACO_FIN ACONDI PLAN'], errors='coerce'))
df_acondi_solidos_consolidated_final['FECHA LIBERACION V0'] = pd.to_datetime(df_acondi_solidos_consolidated_final['FECHA LIBERACION V0_y'], errors='coerce').combine_first(pd.to_datetime(df_acondi_solidos_consolidated_final['FECHA LIBERACION V0'], errors='coerce'))
df_acondi_solidos_consolidated_final['FECHA REQ']           = pd.to_datetime(df_acondi_solidos_consolidated_final['FECHA REQ_y'], errors='coerce').combine_first( pd.to_datetime(df_acondi_solidos_consolidated_final['FECHA REQ'], errors='coerce'))
df_acondi_solidos_consolidated_final['FECHA OTIF']          = pd.to_datetime(df_acondi_solidos_consolidated_final['FECHA OTIF_y'], errors='coerce').combine_first( pd.to_datetime(df_acondi_solidos_consolidated_final['FECHA OTIF'], errors='coerce'))

df_acondi_solidos_consolidated_final['TYPE_BATCH']="E2E"
df_acondi_solidos_consolidated_final["LOTE PT"]=df_acondi_solidos_consolidated_final["LOTE PT"].apply(lambda x: x.strip())
df_acondi_solidos_consolidated_final["LOTE DE GRANEL"]=df_acondi_solidos_consolidated_final["LOTE DE GRANEL"].apply(lambda x: str(x).strip())

df_acondi_solidos_consolidated_final.to_excel(r"C:\Users\x286384\MerckGroup\Digitalización - Documents\DigitalProjectsLibrary\Calibration\Tendencias\OTIF\ConsolidatedDB\PL_Acondi_Solidos_Consolidated.xlsx", index=False)

with pd.ExcelWriter(r"C:\Users\{0}\MerckGroup\Seguimiento Servicio GMS P&G-BP - TrackWise_Deviations\PL_Acondi_Solidos_Consolidated.xlsx".format(muid), engine='xlsxwriter') as writer:
    df_acondi_solidos_consolidated_final.to_excel(writer, index=False, sheet_name='PL_Acondi_Solidos_Consolidated')
    
    # Get the xlsxwriter workbook and worksheet objects
    workbook  = writer.book
    worksheet = writer.sheets['PL_Acondi_Solidos_Consolidated']
    
    # Create a table
    num_rows, num_cols = df_acondi_solidos_consolidated_final.shape
    worksheet.add_table(0, 0, num_rows, num_cols - 1, {'name': 'Acondi_Solidos_Consolidated', 'columns': [{'header': col} for col in df_acondi_solidos_consolidated_final.columns]})






KeyError: "None of [Index(['CLAVE', 'DESCP', 'ORDEN', 'LOTE PT', 'LOTE GRANEL', 'COMENTARIOS',\n       'FIN', 'FECHA REQ CLIENTE', 'OTIF +14', 'FECHA DICTAMEN', 'CANTIDAD',\n       'CANTIDAD C/YIELD', 'LINEA', 'ORDEN'],\n      dtype='str')] are in the [columns]"

In [ ]:
plan_solidos_acondi_LM.keys()

### Liquidos 

In [ ]:
BatchTracker_df.keys()

columns_planning_BT=[ 'SKU', 'DESCRIPCIÓN', 'LOTE PT', 'LOTE DE GRANEL', 'LOTE DE GRANEL 2', 'FECHA REQ', 'FECHA OTIF','FIN FABRI PLAN', 
        'PR_ENTREGA MUESTRA PLAN',  'PR_ENTREGA EXPEDIENTE GR PLAN',
       'ENTREGA ÓRDEN ANALÍTICA PLAN','ACO_FIN ACONDI PLAN',
       'ENTREGA MUESTRA PT PLAN', 'PT_ENTREGA EXPEDIENTE PLAN',
       'PT_ENTREGA EXPEDIENTE REAL','FECHA LIBERACION V0',
       'FECHA LIBERACION ACTUAL', 'TW ID - ALL']

In [ ]:

columns_liquidos_fabri_NE=['CLAVE', 'PRODUCTO', 'LOTE', 'FIN', 'Entrega de muestra', 'Termino Análisis', 'COMENTARIOS',"CANTIDAD", "CANTIDAD C/YIELD", 'ORDEN']
plan_liquidos_fabri_NE.keys()
columns_liquidos_fabri_AM=['CLAVE', 'PRODUCTO', 'LOTE','FIN','Entrega de muestra', 'Termino Análisis','COMENTARIOS',"CANTIDAD", "CANTIDAD C/YIELD", 'RECURSO', 'ORDEN']
plan_liquidos_fabri_AM.keys()
columns_liquidos_fabri_DC=['CLAVE', 'PRODUCTO', 'LOTE',  'FIN', 'Entrega de muestra', 'Termino Análisis','COMENTARIOS', "CANTIDAD", "CANTIDAD C/YIELD", 'RECURSO', 'ORDEN']
plan_liquidos_fabri_DC.keys()

columns_fabri_planning=['SKU', 'DESCRIPCIÓN', 'LOTE DE GRANEL','FIN FABRI PLAN', 'PR_ENTREGA MUESTRA PLAN',  'PR_ENTREGA EXPEDIENTE GR PLAN',
       'ENTREGA ÓRDEN ANALÍTICA PLAN', "CANTIDAD", "CANTIDAD C/YIELD", 'RECURSO', 'ORDEN']

df_liquidos_fabri_NE = clean_rows_based_on_prefix(plan_liquidos_fabri_NE[columns_liquidos_fabri_NE], 'LOTE')
df_liquidos_fabri_AM = clean_rows_based_on_prefix(plan_liquidos_fabri_AM[columns_liquidos_fabri_AM], 'LOTE')
df_liquidos_fabri_DC = clean_rows_based_on_prefix(plan_liquidos_fabri_DC[columns_liquidos_fabri_DC], 'LOTE')

df_liquidos_fabri_NE["RECURSO"]= "Liq Fab"
# Stack the data frames
df_fabri_liquidos_consolidated = pd.concat([df_liquidos_fabri_NE, df_liquidos_fabri_AM, df_liquidos_fabri_DC], ignore_index=True)

# Create a mapping for renaming
rename_mapping = {
    'CLAVE': 'SKU',
    'PRODUCTO': 'DESCRIPCIÓN',
    'LOTE': 'LOTE DE GRANEL',
    'FIN': 'FIN FABRI PLAN',
    'Entrega de muestra': 'PR_ENTREGA MUESTRA PLAN',
    'Termino Análisis': 'ENTREGA ÓRDEN ANALÍTICA PLAN',
    'COMENTARIOS': "PL_STATUS",
    'CANTIDAD': "GR_CANTIDAD",
    "CANTIDAD C/YIELD": "GR_CANTIDAD C/YIELD",
    "ORDEN":"ORDEN GR",
    "RECURSO": "RECURSO GR"
}

# Rename the columns
df_fabri_liquidos_consolidated.rename(columns=rename_mapping, inplace=True)
df_fabri_liquidos_consolidated["PR_ENTREGA EXPEDIENTE GR PLAN"]= pd.to_datetime(df_fabri_liquidos_consolidated["FIN FABRI PLAN"], errors="coerce")+ timedelta(days=7)

df_fabri_liquidos_consolidated["LOTE DE GRANEL"]=df_fabri_liquidos_consolidated["LOTE DE GRANEL"].apply(lambda x: x.strip())

df_fabri_liquidos_consolidated.to_excel(r"C:\Users\x286384\MerckGroup\Digitalización - Documents\DigitalProjectsLibrary\Calibration\Tendencias\OTIF\ConsolidatedDB\PL_Fabri_Liquidos_Consolidated.xlsx", index=False)

with pd.ExcelWriter(r"C:\Users\{0}\MerckGroup\Seguimiento Servicio GMS P&G-BP - TrackWise_Deviations\PL_Fabri_Liquidos_Consolidated.xlsx".format(muid), engine='xlsxwriter') as writer:
    df_fabri_liquidos_consolidated.to_excel(writer, index=False, sheet_name='PL_Fabri_Liquidos_Consolidated')
    
    # Get the xlsxwriter workbook and worksheet objects
    workbook  = writer.book
    worksheet = writer.sheets['PL_Fabri_Liquidos_Consolidated']
    
    # Create a table
    num_rows, num_cols = df_fabri_liquidos_consolidated.shape
    worksheet.add_table(0, 0, num_rows, num_cols - 1, {'name': 'Fabri_Liquidos_Consolidated', 'columns': [{'header': col} for col in df_fabri_liquidos_consolidated.columns]})




In [ ]:
plan_liquidos_acondi_DC.keys()

In [ ]:
columns_liquidos_acondi_AM=[  'CLAVE', 'PRODUCTO', 'LOTE PT', 'LOTE GRANEL 1', 'LOTE GRANEL 2', 'Comentarios', 'FIN', 'FECHA DICTAMEN', 'Fecha de requerimiento','OTIF',"CANTIDAD", "CANTIDAD C/YIELD", "ORDEN"]
plan_liquidos_acondi_AM.keys()
columns_liquidos_acondi_DC=[  'CLAVE', 'PRODUCTO', 'LOTE PT', 'LOTE GRANEL', 'Comentarios', 'FIN', 'FECHA DICTAMEN', 'Fecha de requerimiento','OTIF', "CANTIDAD", "CANTIDAD C/YIELD", "RECURSO", "ORDEN"]
plan_liquidos_acondi_DC.keys()
columns_liquidos_acondi_NE=[  'CLAVE', 'PRODUCTO', 'LOTE PT', 'LOTE GRANEL', 'Comentarios', 'FIN', 'FECHA DICTAMEN', 'Fecha de requerimiento','OTIF', "CANTIDAD", "CANTIDAD C/YIELD", "ORDEN"]
plan_liquidos_acondi_NE.keys()

columns_acondi_planning= ['SKU', 'DESCRIPCIÓN','LOTE PT', 'LOTE DE GRANEL', 'LOTE DE GRANEL 2', 
                           "PL_STATUS", 'ACO_FIN ACONDI PLAN','FECHA LIBERACION V0','FECHA REQ',
                             'FECHA OTIF', "CANTIDAD", "CANTIDAD C/YIELD", "RECURSO", "ORDEN"]

df_liquidos_acondi_NE = clean_rows_based_on_prefix(plan_liquidos_acondi_NE[columns_liquidos_acondi_NE], 'LOTE PT')
df_liquidos_acondi_AM = clean_rows_based_on_prefix(plan_liquidos_acondi_AM[columns_liquidos_acondi_AM], 'LOTE PT')
df_liquidos_acondi_DC = clean_rows_based_on_prefix(plan_liquidos_acondi_DC[columns_liquidos_acondi_DC], 'LOTE PT')

df_liquidos_acondi_AM["RECURSO"]="L1"
df_liquidos_acondi_NE["RECURSO"]="TREN 1"
# Rename "LOTE GRANEL 1" to "LOTE GRANEL" and create "LOTE GRANEL 2" for AM and DC
df_liquidos_acondi_AM.rename(columns={'LOTE GRANEL 1': 'LOTE GRANEL'}, inplace=True)

df_liquidos_acondi_DC['LOTE GRANEL 2'] = pd.NA  # Create empty column
df_liquidos_acondi_NE['LOTE GRANEL 2'] = pd.NA  # Create empty column

# Create a mapping for renaming
rename_mapping = {
    'CLAVE': 'SKU',
    'PRODUCTO': 'DESCRIPCIÓN',
    'LOTE GRANEL': 'LOTE DE GRANEL',
    'LOTE GRANEL 2': 'LOTE DE GRANEL 2',
    'Comentarios': 'PL_STATUS',
    'FIN':  'ACO_FIN ACONDI PLAN',
    'FECHA DICTAMEN': 'FECHA LIBERACION V0',
    'Fecha de requerimiento': 'FECHA REQ',
    'OTIF':'FECHA OTIF',
    'CANTIDAD': "PT_CANTIDAD",
    "CANTIDAD C/YIELD": "PT_CANTIDAD C/YIELD",
    "RECURSO": "RECURSO PT",
    "ORDEN": "ORDEN PT"}

# Rename the columns
df_acondi_liquidos_consolidated = pd.concat([df_liquidos_acondi_NE, df_liquidos_acondi_AM, df_liquidos_acondi_DC], ignore_index=True)

df_acondi_liquidos_consolidated.rename(columns=rename_mapping, inplace=True)
df_acondi_liquidos_consolidated["PT_ENTREGA EXPEDIENTE PLAN"]= pd.to_datetime(df_acondi_liquidos_consolidated["ACO_FIN ACONDI PLAN"], errors="coerce")+ timedelta(days=4)
df_acondi_liquidos_consolidated["LOTE DE GRANEL"]            = df_acondi_liquidos_consolidated["LOTE DE GRANEL"].apply(lambda x: str(x).strip())
df_acondi_liquidos_consolidated["LOTE DE GRANEL 2"]          = df_acondi_liquidos_consolidated["LOTE DE GRANEL 2"].apply(lambda x: str(x).strip())
df_acondi_liquidos_consolidated["LOTE PT"]                   = df_acondi_liquidos_consolidated["LOTE PT"].apply(lambda x: x.strip())
df_acondi_liquidos_consolidated.to_excel(r"C:\Users\x286384\MerckGroup\Digitalización - Documents\DigitalProjectsLibrary\Calibration\Tendencias\OTIF\ConsolidatedDB\PL_Acondi_Liquidos_Consolidated.xlsx", index=False)


with pd.ExcelWriter(r"C:\Users\{0}\MerckGroup\Seguimiento Servicio GMS P&G-BP - TrackWise_Deviations\PL_Acondi_Liquidos_Consolidated.xlsx".format(muid), engine='xlsxwriter') as writer:
    df_acondi_liquidos_consolidated.to_excel(writer, index=False, sheet_name='PL_Acondi_Liquidos_Consolidated')
    
    # Get the xlsxwriter workbook and worksheet objects
    workbook  = writer.book
    worksheet = writer.sheets['PL_Acondi_Liquidos_Consolidated']
    
    # Create a table
    num_rows, num_cols = df_acondi_liquidos_consolidated.shape
    worksheet.add_table(0, 0, num_rows, num_cols - 1, {'name': 'Acondi_Liquidos_Consolidated', 'columns': [{'header': col} for col in df_acondi_liquidos_consolidated.columns]})




### BatchTracker Solids

In [ ]:

df_acondi =df_acondi_solidos_consolidated_final.copy()
df_fabri  =df_fabri_solidos_consolidated.copy()
BT_solidos=merge_fabri_to_acondi(df_acondi=df_acondi,
                                 df_fabri=df_fabri,
                                 family="solids",
                                 liberacion_granel="UD")
BT_solidos["PLAN"]="Solids"

BT_solidos=update_lote_pt(BT_solidos, plan_solidos_fabri_ENV)


BT_solidos.to_excel(r"C:\Users\x286384\MerckGroup\Digitalización - Documents\DigitalProjectsLibrary\Calibration\Tendencias\OTIF\ConsolidatedDB\BT_Solids.xlsx", index=False)

BT_solidos=BT_solidos[['SKU', 'DESCRIPCIÓN', 'LOTE PT', 'LOTE DE GRANEL', 'LOTE DE GRANEL 2', 'PL_STATUS',
        'FECHA LIBERACION V0', 'FECHA REQ', 'FECHA OTIF',
        'FIN FABRI PLAN', 'PR_ENTREGA MUESTRA PLAN', 'PR_ENTREGA EXPEDIENTE GR PLAN', 'ENTREGA ÓRDEN ANALÍTICA PLAN',
        'ACO_FIN ACONDI PLAN','PT_ENTREGA EXPEDIENTE PLAN',
        'TYPE_BATCH', 'BT_analysis', "PLAN", "GR_CANTIDAD C/YIELD", "GR_CANTIDAD", "PT_CANTIDAD C/YIELD", "PT_CANTIDAD",
           "ORDEN GR",  "ORDEN PT", "RECURSO PT", "RECURSO GR"]]



### BatchTracker Liquids

In [ ]:

df_acondi =df_acondi_liquidos_consolidated.copy()
df_fabri  =df_fabri_liquidos_consolidated.copy()
BT_liquidos,_=merge_and_process_batches(df_acondi=df_acondi,
                                 df_fabri=df_fabri)
BT_liquidos["PLAN"]="Liquids"
BT_liquidos.to_excel(r"C:\Users\x286384\MerckGroup\Digitalización - Documents\DigitalProjectsLibrary\Calibration\Tendencias\OTIF\ConsolidatedDB\BT_Liquidos.xlsx", index=False)
    

### BatchTracker Tiroides

In [ ]:

df_acondi =df_acondi_tiroideos_consolidated_final.copy()
df_fabri  =df_fabri_tiroideos_consolidated.copy()
BT_tiroideos=merge_fabri_to_acondi(df_acondi=df_acondi,
                                 df_fabri=df_fabri,
                                 family="tyroids",
                                 liberacion_granel="Dictámen")
BT_tiroideos["PLAN"]="Tyroids"
BT_tiroideos['FECHA LIBERACION V0'] = BT_tiroideos['FECHA LIBERACION V0'].fillna(BT_tiroideos['Dictámen'])
BT_tiroideos.to_excel(r"C:\Users\x286384\MerckGroup\Digitalización - Documents\DigitalProjectsLibrary\Calibration\Tendencias\OTIF\ConsolidatedDB\BT_Tiroideos.xlsx", index=False)



### BatchTracker Consolidation

In [ ]:
print(BT_tiroideos.keys())
print(BT_liquidos.keys())
print(BT_solidos.keys())
BT_liquidos.rename(columns={"PL_STATUS_x": "PL_STATUS"}, inplace=True)
BT_solidos.rename(columns={"PL_STATUS_x": "PL_STATUS"}, inplace=True)

In [ ]:
final_columns_BT =['SKU', 'DESCRIPCIÓN', 'LOTE PT', 'LOTE DE GRANEL', 'LOTE DE GRANEL 2', 'PL_STATUS',
        'FECHA LIBERACION V0', 'FECHA REQ', 'FECHA OTIF',
        'FIN FABRI PLAN', 'PR_ENTREGA MUESTRA PLAN', 'PR_ENTREGA EXPEDIENTE GR PLAN', 'ENTREGA ÓRDEN ANALÍTICA PLAN',
        'ACO_FIN ACONDI PLAN','PT_ENTREGA EXPEDIENTE PLAN',
        'TYPE_BATCH', 'BT_analysis', "PLAN", "GR_CANTIDAD C/YIELD", "GR_CANTIDAD", "PT_CANTIDAD C/YIELD", "PT_CANTIDAD",
          "ORDEN GR", "ORDEN PT", "RECURSO PT", "RECURSO GR"]
final_columns_BT_duple =['SKU', 'DESCRIPCIÓN', 'LOTE PT', 'LOTE DE GRANEL', 'LOTE DE GRANEL 2', 'PL_STATUS',
        'FECHA LIBERACION V0', 'FECHA REQ', 'FECHA OTIF',
        'FIN FABRI PLAN', 'PR_ENTREGA MUESTRA PLAN', 'PR_ENTREGA EXPEDIENTE GR PLAN', 'ENTREGA ÓRDEN ANALÍTICA PLAN',
        'ACO_FIN ACONDI PLAN','PT_ENTREGA EXPEDIENTE PLAN',
        'TYPE_BATCH', 'BT_analysis', "PLAN", "GR_CANTIDAD C/YIELD", "GR_CANTIDAD", "PT_CANTIDAD C/YIELD", "PT_CANTIDAD",
          "ORDEN GR", "ORDEN PT","ORDEN PT", "RECURSO PT", "RECURSO GR"]
BT_tiroideos_month = filter_by_date(BT_tiroideos[final_columns_BT_duple], year=[2025,2026],months=[1,2,3,4,5,6,7,8,9,10,11,12])

BT_liquidos_month = filter_by_date(BT_liquidos[final_columns_BT_duple], year=[2025,2026],months=[1,2,3,4,5,6,7,8,9,10,11,12])

BT_solidos_month = filter_by_date(BT_solidos[final_columns_BT], year=[2025,2026],months=[1,2,3,4,5,6,7,8,9,10,11,12])

print(BT_tiroideos_month.columns)
print(BT_solidos_month.columns)
print(BT_liquidos_month.columns)
BatchTracker_consolidated_plan = pd.concat([BT_tiroideos_month, BT_liquidos_month, BT_solidos_month], ignore_index=True)


In [ ]:



# Load your DataFrame (assuming it's already defined as BatchTracker_consolidated_plan)
BatchTracker_consolidated_plan = BatchTracker_consolidated_plan[['SKU', 'DESCRIPCIÓN', 'LOTE PT', 'LOTE DE GRANEL', 'LOTE DE GRANEL 2', 'PL_STATUS',
        'FECHA LIBERACION V0', 'FECHA REQ', 'FECHA OTIF',
        'FIN FABRI PLAN', 'PR_ENTREGA MUESTRA PLAN', 'PR_ENTREGA EXPEDIENTE GR PLAN', 'ENTREGA ÓRDEN ANALÍTICA PLAN',
        'ACO_FIN ACONDI PLAN','PT_ENTREGA EXPEDIENTE PLAN',
        'TYPE_BATCH', 'BT_analysis', "PLAN", "GR_CANTIDAD C/YIELD", "GR_CANTIDAD", "PT_CANTIDAD C/YIELD", "PT_CANTIDAD",
        "ORDEN GR", "ORDEN PT", "RECURSO GR", "RECURSO PT"]]

BatchTracker_consolidated_plan['SKU'] = BatchTracker_consolidated_plan['SKU'].apply(lambda x: str(x).replace(".",""))



#BatchTracker_consolidated_plan["Planning_Consistency"] = np.where(
#    BatchTracker_consolidated_plan['ENTREGA ÓRDEN ANALÍTICA PLAN'] >= BatchTracker_consolidated_plan['FECHA LIBERACION V0'],
#    'Inconsistente',
#    'Consistente')
# List of date columns to format
date_columns = ['FECHA LIBERACION V0', 'FECHA REQ', 'FECHA OTIF', 'FIN FABRI PLAN', 'PR_ENTREGA MUESTRA PLAN',
                'PR_ENTREGA EXPEDIENTE GR PLAN', 'ENTREGA ÓRDEN ANALÍTICA PLAN', 'ACO_FIN ACONDI PLAN', 'PT_ENTREGA EXPEDIENTE PLAN']

# Formatting the date columns to 'YYYY/MM/DD'
for col in date_columns:
    BatchTracker_consolidated_plan[col] = pd.to_datetime(BatchTracker_consolidated_plan[col], errors='coerce').dt.normalize()

# Save the updated DataFrame to Excel
BatchTracker_consolidated_plan.to_excel(r"C:\Users\x286384\MerckGroup\Digitalización - Documents\DigitalProjectsLibrary\Calibration\Tendencias\OTIF\ConsolidatedDB\BatchTracker_allV0_today.xlsx", index=False)

# Save DataFrame to Excel file with table formatting
with pd.ExcelWriter(r"C:\Users\{0}\MerckGroup\Seguimiento Servicio GMS P&G-BP - TrackWise_Deviations\BT_PLAN_Consolidated_allV0.xlsx".format(muid), engine='xlsxwriter') as writer:
    BatchTracker_consolidated_plan.to_excel(writer, index=False, sheet_name='BT_consolidates')
    # Get the xlsxwriter workbook and worksheet objects
    workbook  = writer.book
    worksheet = writer.sheets['BT_consolidates']
    
    # Create a table
    num_rows, num_cols = BatchTracker_consolidated_plan.shape
    worksheet.add_table(0, 0, num_rows, num_cols - 1, {'name': 'DataTable', 'columns': [{'header': col} for col in BatchTracker_consolidated_plan.columns]})



In [ ]:
combined_Fabri_df

#### Master BatchTracker for Data Analysis

In [ ]:
#estaba como numero normal, no como string el 1 y lanzaba un error de tipo de dato
BatchTracker_consolidated_plan["PR_Estatus"]="1"
BatchTracker_consolidated_plan["QC_Estatus"]="1"
# Function to populate BatchTracker_df based on matching LOTES
for index, row in combined_Fabri_df.iterrows():
    # Check if 'LOTE DE GRANEL' from combined_Fabri_df is in any LOTES of BatchTracker_df
    if row['LOTE DE GRANEL'] in BatchTracker_consolidated_plan['LOTE PT'].values or \
       row['LOTE DE GRANEL'] in BatchTracker_consolidated_plan['LOTE DE GRANEL'].values or \
       row['LOTE DE GRANEL'] in BatchTracker_consolidated_plan['LOTE DE GRANEL 2'].values:
        
        # Update BatchTracker_df with values from combined_Fabri_df
        BatchTracker_consolidated_plan.loc[BatchTracker_consolidated_plan['LOTE PT']          == row['LOTE DE GRANEL'], 'FIN FABRI REAL'] = row['FIN FABRI REAL']
        BatchTracker_consolidated_plan.loc[BatchTracker_consolidated_plan['LOTE DE GRANEL 2'] == row['LOTE DE GRANEL'], 'FIN FABRI REAL'] = row['FIN FABRI REAL']
        BatchTracker_consolidated_plan.loc[BatchTracker_consolidated_plan['LOTE DE GRANEL']   == row['LOTE DE GRANEL'], 'FIN FABRI REAL'] = row['FIN FABRI REAL']

        # Update BatchTracker_df with values from combined_Fabri_df
        BatchTracker_consolidated_plan.loc[BatchTracker_consolidated_plan['LOTE PT']          == row['LOTE DE GRANEL'], "PR_Estatus"] = row["PR_Estatus"]
        BatchTracker_consolidated_plan.loc[BatchTracker_consolidated_plan['LOTE DE GRANEL 2'] == row['LOTE DE GRANEL'], "PR_Estatus"] = row["PR_Estatus"]
        BatchTracker_consolidated_plan.loc[BatchTracker_consolidated_plan['LOTE DE GRANEL']   == row['LOTE DE GRANEL'], "PR_Estatus"] = row["PR_Estatus"]

        BatchTracker_consolidated_plan.loc[BatchTracker_consolidated_plan['LOTE PT']          == row['LOTE DE GRANEL'], 'PR_ENTREGA EXPEDIENTE GR REAL'] = row['PR_ENTREGA EXPEDIENTE GR REAL']
        BatchTracker_consolidated_plan.loc[BatchTracker_consolidated_plan['LOTE DE GRANEL 2'] == row['LOTE DE GRANEL'], 'PR_ENTREGA EXPEDIENTE GR REAL'] = row['PR_ENTREGA EXPEDIENTE GR REAL']
        BatchTracker_consolidated_plan.loc[BatchTracker_consolidated_plan['LOTE DE GRANEL']   == row['LOTE DE GRANEL'], 'PR_ENTREGA EXPEDIENTE GR REAL'] = row['PR_ENTREGA EXPEDIENTE GR REAL']

# Function to populate BatchTracker_df based on matching LOTES
for index, row in combined_acondi_df.iterrows():
    # Check if 'LOTE DE GRANEL' from combined_Fabri_df is in any LOTES of BatchTracker_df
    if row['LOTE PT'] in BatchTracker_consolidated_plan['LOTE PT'].values or \
       row['LOTE PT'] in BatchTracker_consolidated_plan['LOTE DE GRANEL'].values or \
       row['LOTE PT'] in BatchTracker_consolidated_plan['LOTE DE GRANEL 2'].values:
        
        # Update BatchTracker_df with values from combined_Fabri_df
        BatchTracker_consolidated_plan.loc[BatchTracker_consolidated_plan['LOTE PT']           == row['LOTE PT'], 'ACO_FIN ACONDI REAL'] = row['ACO_FIN ACONDI REAL']
        BatchTracker_consolidated_plan.loc[BatchTracker_consolidated_plan['LOTE DE GRANEL 2']  == row['LOTE PT'], 'ACO_FIN ACONDI REAL'] = row['ACO_FIN ACONDI REAL']
        BatchTracker_consolidated_plan.loc[BatchTracker_consolidated_plan['LOTE DE GRANEL']    == row['LOTE PT'], 'ACO_FIN ACONDI REAL'] = row['ACO_FIN ACONDI REAL']

        BatchTracker_consolidated_plan.loc[BatchTracker_consolidated_plan['LOTE PT']           == row['LOTE PT'], 'PT_ENTREGA EXPEDIENTE REAL'] =  pd.to_datetime(row['PT_ENTREGA EXPEDIENTE REAL']).date()
        print(row['LOTE PT'], pd.to_datetime(row['PT_ENTREGA EXPEDIENTE REAL']).date())

# Function to populate BatchTracker_df based on matching LOTES
for index, row in combined_qc_df.iterrows():
    # Check if 'LOTE DE GRANEL' from combined_Fabri_df is in any LOTES of BatchTracker_df
    if row['LOTE'] in BatchTracker_consolidated_plan['LOTE PT'].values or \
       row['LOTE'] in BatchTracker_consolidated_plan['LOTE DE GRANEL'].values or \
       row['LOTE'] in BatchTracker_consolidated_plan['LOTE DE GRANEL 2'].values:
        
        # Update BatchTracker_df with values from combined_Fabri_df
        BatchTracker_consolidated_plan.loc[BatchTracker_consolidated_plan['LOTE PT']           == row['LOTE'], 'PR_ENTREGA MUESTRA REAL'] = row['PR_ENTREGA MUESTRA REAL']
        BatchTracker_consolidated_plan.loc[BatchTracker_consolidated_plan['LOTE DE GRANEL 2']  == row['LOTE'], 'PR_ENTREGA MUESTRA REAL'] = row['PR_ENTREGA MUESTRA REAL']
        BatchTracker_consolidated_plan.loc[BatchTracker_consolidated_plan['LOTE DE GRANEL']    == row['LOTE'], 'PR_ENTREGA MUESTRA REAL'] = row['PR_ENTREGA MUESTRA REAL']

        # Update BatchTracker_df with values from combined_Fabri_df
        BatchTracker_consolidated_plan.loc[BatchTracker_consolidated_plan['LOTE PT']           == row['LOTE'], "QC_Estatus"] = row["QC_Estatus"]
        BatchTracker_consolidated_plan.loc[BatchTracker_consolidated_plan['LOTE DE GRANEL 2']  == row['LOTE'], "QC_Estatus"] = row["QC_Estatus"]
        BatchTracker_consolidated_plan.loc[BatchTracker_consolidated_plan['LOTE DE GRANEL']    == row['LOTE'], "QC_Estatus"] = row["QC_Estatus"]

        BatchTracker_consolidated_plan.loc[BatchTracker_consolidated_plan['LOTE PT']           == row['LOTE'], 'ENTREGA ÓRDEN ANALÍTICA REAL'] = row['ENTREGA ÓRDEN ANALÍTICA REAL']
        BatchTracker_consolidated_plan.loc[BatchTracker_consolidated_plan['LOTE DE GRANEL 2']  == row['LOTE'], 'ENTREGA ÓRDEN ANALÍTICA REAL'] = row['ENTREGA ÓRDEN ANALÍTICA REAL']
        BatchTracker_consolidated_plan.loc[BatchTracker_consolidated_plan['LOTE DE GRANEL']    == row['LOTE'], 'ENTREGA ÓRDEN ANALÍTICA REAL'] = row['ENTREGA ÓRDEN ANALÍTICA REAL']


In [ ]:

# Apply the function to each row of BatchTracker_df and create the new column
BatchTracker_consolidated_plan['Deviation_Info'] = BatchTracker_consolidated_plan.apply(create_deviation_info, axis=1)

# Apply the function to each row of BatchTracker_df and create the new column 'Deviation Due Date'
BatchTracker_consolidated_plan['Deviation Due Date'] = BatchTracker_consolidated_plan.apply(create_deviation_due_date, axis=1)

# Apply the function to each row of BatchTracker_df and create the new column 'Deviation Due Date'
BatchTracker_consolidated_plan['Deviation Close Date'] = BatchTracker_consolidated_plan.apply(create_deviation_closure_date, axis=1)
# Convert the new columns to datetime
BatchTracker_consolidated_plan['Deviation Due Date'] = pd.to_datetime(BatchTracker_consolidated_plan['Deviation Due Date'], errors='coerce',dayfirst=True)
BatchTracker_consolidated_plan['Deviation Close Date'] = pd.to_datetime(BatchTracker_consolidated_plan['Deviation Close Date'], errors='coerce',dayfirst=True)

BatchTracker_consolidated_plan = BatchTracker_consolidated_plan[['SKU', 'DESCRIPCIÓN', 'LOTE PT', 'LOTE DE GRANEL', 'LOTE DE GRANEL 2',
       'PL_STATUS', 'FECHA LIBERACION V0', 'FECHA REQ', 'FECHA OTIF',
       'FIN FABRI PLAN', 'FIN FABRI REAL', 
       'PR_ENTREGA MUESTRA PLAN',  'PR_ENTREGA MUESTRA REAL',
       'PR_ENTREGA EXPEDIENTE GR PLAN', 'PR_ENTREGA EXPEDIENTE GR REAL',
       'ENTREGA ÓRDEN ANALÍTICA PLAN', 'ENTREGA ÓRDEN ANALÍTICA REAL',
       'ACO_FIN ACONDI PLAN', 'ACO_FIN ACONDI REAL',
       'PT_ENTREGA EXPEDIENTE PLAN', 'PT_ENTREGA EXPEDIENTE REAL',
        'TYPE_BATCH',
       'BT_analysis', 'PLAN', 'GR_CANTIDAD C/YIELD', 'GR_CANTIDAD',
       'PT_CANTIDAD C/YIELD', 'PT_CANTIDAD', 'PR_Estatus', 'QC_Estatus',
       'Deviation_Info', 'Deviation Due Date','Deviation Close Date']]


# Formatting the date columns to 'YYYY/MM/DD'
for col in date_columns:
    BatchTracker_consolidated_plan[col] = pd.to_datetime(BatchTracker_consolidated_plan[col], errors='coerce').dt.normalize()
with pd.ExcelWriter(r"C:\Users\{0}\MerckGroup\Seguimiento Servicio GMS P&G-BP - TrackWise_Deviations\MasterBT_Consolidated.xlsx".format(muid), engine='xlsxwriter') as writer:
    BatchTracker_consolidated_plan.to_excel(writer, index=False, sheet_name='BT_consolidates')
    
    # Get the xlsxwriter workbook and worksheet objects
    workbook  = writer.book
    worksheet = writer.sheets['BT_consolidates']
    
    # Create a table
    num_rows, num_cols = BatchTracker_consolidated_plan.shape
    worksheet.add_table(0, 0, num_rows, num_cols - 1, {'name': 'DataTable', 'columns': [{'header': col} for col in BatchTracker_consolidated_plan.columns]})



BatchTracker_consolidated_plan.to_excel("MasterBatchTracker.xlsx")


## Concatenación QA

# Building BatchTracker

In [ ]:
#estaba como numero normal, no como string el 1 y lanzaba un error de tipo de dato
BatchTracker_df["PR_Estatus"]="1"
BatchTracker_df["QC_Estatus"]="1"

In [ ]:
BatchTracker_df

In [ ]:


# Function to populate BatchTracker_df based on matching LOTES
for index, row in combined_Fabri_df.iterrows():
    # Check if 'LOTE DE GRANEL' from combined_Fabri_df is in any LOTES of BatchTracker_df
    if row['LOTE DE GRANEL'] in BatchTracker_df['LOTE PT'].values or \
       row['LOTE DE GRANEL'] in BatchTracker_df['LOTE DE GRANEL'].values or \
       row['LOTE DE GRANEL'] in BatchTracker_df['LOTE DE GRANEL 2'].values:
        
        # Update BatchTracker_df with values from combined_Fabri_df
        BatchTracker_df.loc[BatchTracker_df['LOTE PT']          == row['LOTE DE GRANEL'], 'FIN FABRI REAL'] = row['FIN FABRI REAL']
        BatchTracker_df.loc[BatchTracker_df['LOTE DE GRANEL 2'] == row['LOTE DE GRANEL'], 'FIN FABRI REAL'] = row['FIN FABRI REAL']
        BatchTracker_df.loc[BatchTracker_df['LOTE DE GRANEL']   == row['LOTE DE GRANEL'], 'FIN FABRI REAL'] = row['FIN FABRI REAL']

        # Update BatchTracker_df with values from combined_Fabri_df
        BatchTracker_df.loc[BatchTracker_df['LOTE PT']          == row['LOTE DE GRANEL'], "PR_Estatus"] = row["PR_Estatus"]
        BatchTracker_df.loc[BatchTracker_df['LOTE DE GRANEL 2'] == row['LOTE DE GRANEL'], "PR_Estatus"] = row["PR_Estatus"]
        BatchTracker_df.loc[BatchTracker_df['LOTE DE GRANEL']   == row['LOTE DE GRANEL'], "PR_Estatus"] = row["PR_Estatus"]

        BatchTracker_df.loc[BatchTracker_df['LOTE PT']          == row['LOTE DE GRANEL'], 'PR_ENTREGA EXPEDIENTE GR REAL'] = row['PR_ENTREGA EXPEDIENTE GR REAL']
        BatchTracker_df.loc[BatchTracker_df['LOTE DE GRANEL 2'] == row['LOTE DE GRANEL'], 'PR_ENTREGA EXPEDIENTE GR REAL'] = row['PR_ENTREGA EXPEDIENTE GR REAL']
        BatchTracker_df.loc[BatchTracker_df['LOTE DE GRANEL']   == row['LOTE DE GRANEL'], 'PR_ENTREGA EXPEDIENTE GR REAL'] = row['PR_ENTREGA EXPEDIENTE GR REAL']

# Function to populate BatchTracker_df based on matching LOTES
for index, row in combined_acondi_df.iterrows():
    # Check if 'LOTE DE GRANEL' from combined_Fabri_df is in any LOTES of BatchTracker_df
    if row['LOTE PT'] in BatchTracker_df['LOTE PT'].values or \
       row['LOTE PT'] in BatchTracker_df['LOTE DE GRANEL'].values or \
       row['LOTE PT'] in BatchTracker_df['LOTE DE GRANEL 2'].values:
        
        # Update BatchTracker_df with values from combined_Fabri_df
        BatchTracker_df.loc[BatchTracker_df['LOTE PT']           == row['LOTE PT'], 'ACO_FIN ACONDI REAL'] = row['ACO_FIN ACONDI REAL']
        BatchTracker_df.loc[BatchTracker_df['LOTE DE GRANEL 2']  == row['LOTE PT'], 'ACO_FIN ACONDI REAL'] = row['ACO_FIN ACONDI REAL']
        BatchTracker_df.loc[BatchTracker_df['LOTE DE GRANEL']    == row['LOTE PT'], 'ACO_FIN ACONDI REAL'] = row['ACO_FIN ACONDI REAL']

        BatchTracker_df.loc[BatchTracker_df['LOTE PT']           == row['LOTE PT'], 'PT_ENTREGA EXPEDIENTE REAL'] =  pd.to_datetime(row['PT_ENTREGA EXPEDIENTE REAL']).date()
        print(row['LOTE PT'], pd.to_datetime(row['PT_ENTREGA EXPEDIENTE REAL']).date())

# Function to populate BatchTracker_df based on matching LOTES
for index, row in combined_qc_df.iterrows():
    # Check if 'LOTE DE GRANEL' from combined_Fabri_df is in any LOTES of BatchTracker_df
    if row['LOTE'] in BatchTracker_df['LOTE PT'].values or \
       row['LOTE'] in BatchTracker_df['LOTE DE GRANEL'].values or \
       row['LOTE'] in BatchTracker_df['LOTE DE GRANEL 2'].values:
        
        # Update BatchTracker_df with values from combined_Fabri_df
        #BatchTracker_df.loc[BatchTracker_df['LOTE PT']           == row['LOTE'], 'PR_ENTREGA MUESTRA REAL'] = row['PR_ENTREGA MUESTRA REAL']
        #BatchTracker_df.loc[BatchTracker_df['LOTE DE GRANEL 2']  == row['LOTE'], 'PR_ENTREGA MUESTRA REAL'] = row['PR_ENTREGA MUESTRA REAL']
        BatchTracker_df.loc[BatchTracker_df['LOTE DE GRANEL']    == row['LOTE'], 'PR_ENTREGA MUESTRA REAL'] = row['PR_ENTREGA MUESTRA REAL']

        # Update BatchTracker_df with values from combined_Fabri_df
        #BatchTracker_df.loc[BatchTracker_df['LOTE PT']           == row['LOTE'], "QC_Estatus"] = row["QC_Estatus"]
        #BatchTracker_df.loc[BatchTracker_df['LOTE DE GRANEL 2']  == row['LOTE'], "QC_Estatus"] = row["QC_Estatus"]
        BatchTracker_df.loc[BatchTracker_df['LOTE DE GRANEL']    == row['LOTE'], "QC_Estatus"] = row["QC_Estatus"]

       #BatchTracker_df.loc[BatchTracker_df['LOTE PT']           == row['LOTE'], 'ENTREGA ÓRDEN ANALÍTICA REAL'] = row['ENTREGA ÓRDEN ANALÍTICA REAL']
       #BatchTracker_df.loc[BatchTracker_df['LOTE DE GRANEL 2']  == row['LOTE'], 'ENTREGA ÓRDEN ANALÍTICA REAL'] = row['ENTREGA ÓRDEN ANALÍTICA REAL']
        BatchTracker_df.loc[BatchTracker_df['LOTE DE GRANEL']    == row['LOTE'], 'ENTREGA ÓRDEN ANALÍTICA REAL'] = row['ENTREGA ÓRDEN ANALÍTICA REAL']



In [ ]:
BatchTracker_df.to_excel("Dummy_batch_tracker.xlsx")

In [ ]:
# Save DataFrame to Excel file with table formatting
with pd.ExcelWriter(r"C:\Users\{0}\MerckGroup\Seguimiento Servicio GMS P&G-BP - TrackWise_Deviations\Consolidated_BT_Deviations.xlsx".format(muid), engine='xlsxwriter') as writer:
    BatchTracker_df.to_excel(writer, index=False, sheet_name='TW')
    
    # Get the xlsxwriter workbook and worksheet objects
    workbook  = writer.book
    worksheet = writer.sheets['TW']
    
    # Create a table
    num_rows, num_cols = BatchTracker_df.shape
    worksheet.add_table(0, 0, num_rows, num_cols - 1, {'name': 'DataTable', 'columns': [{'header': col} for col in BatchTracker_df.columns]})


In [ ]:
BatchTracker_df.keys()

In [ ]:
fin = time.perf_counter()
tiempo_total = fin - inicio
minutos, segundos = divmod(tiempo_total, 60)
print(
    f"Tiempo total de ejecución: "
    f"{int(minutos)} minutos y {segundos:.2f} segundos")